# AeroPredict — Modelado predictivo del TFM

**Notebook limpio de ejecución en R / Google Colab**

Este notebook contiene el flujo principal del modelado a partir de `gold.feature_store` en PostgreSQL/Neon. Está organizado para ejecutarse de arriba abajo y evita mezclar pruebas exploratorias anteriores con la evaluación final.

## Objetivo

1. Clasificar vuelos cuyo retraso estimado sea de **15 minutos o más** (`RETRASO`) frente al resto (`NO_RETRASO`).
2. Para los vuelos de la clase `RETRASO`, estimar los minutos mediante **regresión lineal**, el modelo de trabajo acordado para esta fase.
3. Seleccionar el umbral de clasificación usando VALIDACIÓN y reservar TEST FINAL para informar las métricas finales.

> **Limitación de los datos:** `retraso_minutos` se calcula como `estimated_local - scheduled_local`. Con los datos actuales representa la diferencia entre la hora estimada y la programada en el momento de captura; no equivale necesariamente al retraso real observado al salir o aterrizar. Las métricas deben interpretarse con esta limitación.

## 1. Preparar el entorno

Instalamos y cargamos únicamente las librerías necesarias para consultar PostgreSQL, preparar los datos y entrenar los modelos.

In [ ]:
paquetes <- c("DBI", "RPostgres", "dplyr", "httr", "getPass", "gbm")
instalados <- rownames(installed.packages())
faltantes <- setdiff(paquetes, instalados)

if (length(faltantes) > 0) {
  install.packages(faltantes, repos = "https://cloud.r-project.org")
}

library(DBI)
library(RPostgres)
library(dplyr)
library(gbm)

set.seed(123)
cat("Entorno preparado.\n")

Installing packages into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union


Loaded gbm 2.3.1

This version of gbm is no longer under development. Consider transitioning to gbm3, https://github.com/gbm-developers/gbm3



Entorno preparado.


## 2. Conectar a PostgreSQL / Neon de forma segura

La URI se solicita mediante entrada oculta y no se imprime en la salida. También puede definirse previamente como variable de entorno `AEROPREDICT_DATABASE_URL`. No guardes una URI con contraseña en el código ni en las salidas del notebook.


ACA VA LA CLAVE

In [ ]:
if (nzchar(Sys.getenv("AEROPREDICT_DATABASE_URL"))) {
  postgres_uri <- Sys.getenv("AEROPREDICT_DATABASE_URL")
} else {
  postgres_uri <- getPass::getPass(
    "Pega la URI completa de PostgreSQL/Neon (entrada oculta): "
  )
}

if (!nzchar(postgres_uri)) {
  stop("No se proporcionó la URI de PostgreSQL.")
}

partes_uri <- httr::parse_url(postgres_uri)
if (is.null(partes_uri$hostname) || is.null(partes_uri$username) ||
    is.null(partes_uri$password) || is.null(partes_uri$path)) {
  stop("La URI no contiene host, usuario, contraseña y nombre de base de datos.")
}

puerto_pg <- if (is.null(partes_uri$port)) 5432L else as.integer(partes_uri$port)
base_pg <- sub("^/", "", partes_uri$path)

con <- DBI::dbConnect(
  RPostgres::Postgres(),
  host = partes_uri$hostname,
  port = puerto_pg,
  dbname = base_pg,
  user = utils::URLdecode(partes_uri$username),
  password = utils::URLdecode(partes_uri$password),
  sslmode = "require"
)

if (!DBI::dbIsValid(con)) {
  stop("No se pudo validar la conexión a PostgreSQL.")
}

# Eliminar de la memoria de trabajo la cadena de conexión una vez conectados.
rm(postgres_uri, partes_uri, puerto_pg, base_pg)
cat("Conexión segura validada. La URI no se ha mostrado.\n")
DBI::dbGetQuery(con, "SELECT 1 AS conexion_ok")

Pega la URI completa de PostgreSQL/Neon (entrada oculta): ··········
Conexión segura validada. La URI no se ha mostrado.


conexion_ok
<int>
1


## 3. Cargar los datos y definir el objetivo

Se consulta la tabla Gold hasta la fecha de corte acordada (**19 de octubre de 2026**). La etiqueta se define como `1` cuando `retraso_minutos >= 15`; de lo contrario es `0`. Las variables de identificación y fecha se conservan para auditoría, pero no se utilizan como predictores.

In [ ]:
fecha_corte_datos <- as.Date("2026-10-19")

variables_predictoras <- c(
  "hora_vuelo", "dia_semana", "airline_iata", "other_airport_iata",
  "temperatura_metar", "punto_rocio_metar", "relh"
)

variables_categoricas <- c(
  "hora_vuelo", "dia_semana", "airline_iata", "other_airport_iata"
)

sql_modelado <- "
SELECT
    aena_airport_iata,
    flight_number,
    flight_type,
    scheduled_local,
    hora_vuelo,
    dia_semana,
    airline_iata,
    other_airport_iata,
    temperatura_metar,
    punto_rocio_metar,
    relh,
    retraso_minutos
FROM gold.feature_store
WHERE DATE(scheduled_local) <= DATE '2026-10-19'
  AND retraso_minutos IS NOT NULL
"

datos_brutos <- DBI::dbGetQuery(con, sql_modelado)

if (nrow(datos_brutos) == 0) {
  stop("La consulta no devolvió registros para modelar.")
}

df_modelado <- datos_brutos %>%
  mutate(
    fecha_vuelo = as.Date(substr(as.character(scheduled_local), 1, 10)),
    retraso_minutos = as.numeric(as.character(retraso_minutos)),
    temperatura_metar = as.numeric(as.character(temperatura_metar)),
    punto_rocio_metar = as.numeric(as.character(punto_rocio_metar)),
    relh = as.numeric(as.character(relh)),
    across(all_of(variables_categoricas), as.character),
    retraso_15_min = as.integer(retraso_minutos >= 15)
  ) %>%
  arrange(fecha_vuelo)

filas_antes_limpieza <- nrow(df_modelado)
columnas_modelado <- c(variables_predictoras, "fecha_vuelo", "retraso_minutos", "retraso_15_min")
filas_completas <- complete.cases(df_modelado[, columnas_modelado])
filas_excluidas_na <- sum(!filas_completas)
df_modelado <- df_modelado[filas_completas, , drop = FALSE]

if (nrow(df_modelado) == 0) {
  stop("No quedan filas completas tras revisar predictores y objetivo.")
}
if (anyNA(df_modelado$fecha_vuelo)) {
  stop("Hay fechas de vuelo que no se pudieron interpretar.")
}

cat("Registros consultados:", filas_antes_limpieza, "\n")
cat("Registros excluidos por valores ausentes/no válidos:", filas_excluidas_na, "\n")
cat("Registros utilizables:", nrow(df_modelado), "\n")
cat("Periodo:", as.character(min(df_modelado$fecha_vuelo)), "a",
    as.character(max(df_modelado$fecha_vuelo)), "\n")
cat("Distribución del objetivo (0 = <15 min; 1 = >=15 min):\n")
print(table(df_modelado$retraso_15_min))
cat("Resumen de retraso_minutos:\n")
print(summary(df_modelado$retraso_minutos))

Registros consultados: 83407 
Registros excluidos por valores ausentes/no válidos: 0 
Registros utilizables: 83407 
Periodo: 2026-08-01 a 2026-10-19 
Distribución del objetivo (0 = <15 min; 1 = >=15 min):

    0     1 
70341 13066 
Resumen de retraso_minutos:
    Min.  1st Qu.   Median     Mean  3rd Qu.     Max. 
 -21.000    0.000    0.000    9.016    5.000 1440.000 


## 4. Controles esenciales de calidad

Se conservan comprobaciones que ayudan a interpretar el conjunto: duplicados por clave lógica y distribución del objetivo. No se modifican ni se escriben datos en las tablas Gold.

In [ ]:
sql_duplicados <- "
SELECT
    COUNT(*) AS total_registros,
    COUNT(DISTINCT (
        aena_airport_iata,
        UPPER(TRIM(flight_number)),
        LOWER(TRIM(flight_type)),
        scheduled_local
    )) AS vuelos_unicos
FROM gold.feature_store
WHERE DATE(scheduled_local) <= DATE '2026-10-19'
  AND retraso_minutos IS NOT NULL;
"

control_duplicados <- DBI::dbGetQuery(con, sql_duplicados)
cat("Control de duplicados por clave lógica:\n")
print(control_duplicados)

cat("\nProporción de retrasos estimados >=15 minutos:\n")
print(round(prop.table(table(df_modelado$retraso_15_min)), 4))

DBI::dbDisconnect(con)
cat("Conexión a PostgreSQL cerrada; el modelado continúa con los datos cargados en memoria.\n")

Control de duplicados por clave lógica:
  total_registros vuelos_unicos
1           83407         83407

Proporción de retrasos estimados >=15 minutos:

     0      1 
0.8433 0.1567 
Conexión a PostgreSQL cerrada; el modelado continúa con los datos cargados en memoria.


## 5. Separación temporal en TRAIN, VALIDACIÓN y TEST FINAL

Las fechas distintas se ordenan cronológicamente. Aproximadamente el 60 % de las fechas se asigna a TRAIN, el siguiente 20 % a VALIDACIÓN y el tramo más reciente a TEST FINAL. La partición se hace por fecha, no al azar, para respetar el orden temporal.

- **TRAIN:** aprende los parámetros del modelo y del preprocesamiento.
- **VALIDACIÓN:** selecciona el umbral de clasificación.
- **TEST FINAL:** se usa una sola vez en este flujo para presentar las métricas finales.

In [ ]:
fechas <- sort(unique(df_modelado$fecha_vuelo))
if (length(fechas) < 10) {
  stop("No hay suficientes fechas distintas para una separación temporal.")
}

corte_validacion <- fechas[floor(length(fechas) * 0.60)]
corte_test <- fechas[floor(length(fechas) * 0.80)]

if (corte_validacion >= corte_test) {
  stop("Las fechas de corte no permiten separar TRAIN, VALIDACIÓN y TEST.")
}

datos_train <- df_modelado %>% filter(fecha_vuelo < corte_validacion)
datos_val <- df_modelado %>%
  filter(fecha_vuelo >= corte_validacion, fecha_vuelo < corte_test)
datos_test_final <- df_modelado %>% filter(fecha_vuelo >= corte_test)

if (nrow(datos_train) == 0 || nrow(datos_val) == 0 || nrow(datos_test_final) == 0) {
  stop("Uno de los conjuntos temporales quedó vacío.")
}

resumen_particion <- data.frame(
  conjunto = c("TRAIN", "VALIDACIÓN", "TEST FINAL"),
  filas = c(nrow(datos_train), nrow(datos_val), nrow(datos_test_final)),
  desde = c(min(datos_train$fecha_vuelo), min(datos_val$fecha_vuelo), min(datos_test_final$fecha_vuelo)),
  hasta = c(max(datos_train$fecha_vuelo), max(datos_val$fecha_vuelo), max(datos_test_final$fecha_vuelo))
)
print(resumen_particion, row.names = FALSE)

cat("\nDistribución de la etiqueta por conjunto:\n")
print(dplyr::bind_rows(
  datos_train %>% count(retraso_15_min) %>% mutate(conjunto = "TRAIN"),
  datos_val %>% count(retraso_15_min) %>% mutate(conjunto = "VALIDACIÓN"),
  datos_test_final %>% count(retraso_15_min) %>% mutate(conjunto = "TEST FINAL")
) %>% select(conjunto, retraso_15_min, n))

   conjunto filas      desde      hasta
      TRAIN 44495 2026-08-01 2026-09-22
 VALIDACIÓN 14173 2026-09-23 2026-10-12
 TEST FINAL 24739 2026-10-13 2026-10-19

Distribución de la etiqueta por conjunto:
    conjunto retraso_15_min     n
1      TRAIN              0 38304
2      TRAIN              1  6191
3 VALIDACIÓN              0 13282
4 VALIDACIÓN              1   891
5 TEST FINAL              0 18755
6 TEST FINAL              1  5984


## 6. Preprocesamiento aprendido solo con TRAIN

Las categorías de aerolínea y aeropuerto poco frecuentes se agrupan en `OTRA`, utilizando únicamente frecuencias de TRAIN. Los valores temporales (`hora_vuelo` y `dia_semana`) deben pertenecer a categorías ya vistas en TRAIN; si aparece una categoría temporal desconocida, el código se detiene para evitar convertirla silenciosamente en un valor incorrecto.

Las matrices de variables indicadoras se alinean con las columnas aprendidas en TRAIN. Las columnas constantes se eliminan antes del entrenamiento.

In [ ]:
preparar_categorias_temporal <- function(train, validacion, test,
                                        variables_categoricas,
                                        variables_agrupadas = c("airline_iata", "other_airport_iata"),
                                        frecuencia_minima = 2) {
  conjuntos <- list(train = train, validacion = validacion, test = test)
  reglas <- list()

  # Variables de aerolínea/aeropuerto: agrupar categorías poco frecuentes según TRAIN.
  for (v in intersect(variables_agrupadas, variables_categoricas)) {
    valores_train <- as.character(conjuntos$train[[v]])
    frecuencias <- table(valores_train[!is.na(valores_train)])
    niveles_frecuentes <- names(frecuencias[frecuencias >= frecuencia_minima])

    if (length(niveles_frecuentes) == 0 && length(frecuencias) > 0) {
      niveles_frecuentes <- names(frecuencias)[which.max(frecuencias)]
    }

    niveles_finales <- unique(c(niveles_frecuentes, "OTRA"))
    for (nombre_conjunto in names(conjuntos)) {
      valores <- as.character(conjuntos[[nombre_conjunto]][[v]])
      valores[is.na(valores) | !(valores %in% niveles_frecuentes)] <- "OTRA"
      conjuntos[[nombre_conjunto]][[v]] <- factor(valores, levels = niveles_finales)
    }

    reglas[[v]] <- list(
      tipo = "frecuencia",
      niveles_frecuentes = niveles_frecuentes,
      niveles_finales = niveles_finales,
      frecuencia_minima = frecuencia_minima
    )
  }

  # Variables temporales: solo se admiten niveles observados en TRAIN.
  for (v in setdiff(variables_categoricas, variables_agrupadas)) {
    niveles_train <- sort(unique(as.character(conjuntos$train[[v]])))
    for (nombre_conjunto in names(conjuntos)) {
      valores <- as.character(conjuntos[[nombre_conjunto]][[v]])
      valores_desconocidos <- !is.na(valores) & !(valores %in% niveles_train)
      if (any(valores_desconocidos)) {
        stop(paste0("La variable ", v,
                    " contiene categorías no vistas en TRAIN en el conjunto ",
                    nombre_conjunto, ". Revisar esos valores antes de continuar."))
      }
      conjuntos[[nombre_conjunto]][[v]] <- factor(valores, levels = niveles_train)
    }
    reglas[[v]] <- list(tipo = "niveles_train", niveles_finales = niveles_train)
  }

  list(
    train = conjuntos$train,
    validacion = conjuntos$validacion,
    test = conjuntos$test,
    reglas = reglas
  )
}

crear_matriz_predictores <- function(datos, formula_x, columnas = NULL) {
  X <- stats::model.matrix(formula_x, data = datos)
  X <- X[, colnames(X) != "(Intercept)", drop = FALSE]

  if (is.null(columnas)) {
    if (ncol(X) == 0) stop("No se generaron variables indicadoras.")
    constantes <- vapply(seq_len(ncol(X)), function(j) {
      length(unique(X[, j])) <= 1
    }, logical(1))
    X <- X[, !constantes, drop = FALSE]
    return(X)
  }

  X_alineada <- matrix(
    0,
    nrow = nrow(datos),
    ncol = length(columnas),
    dimnames = list(NULL, columnas)
  )
  comunes <- intersect(columnas, colnames(X))
  X_alineada[, comunes] <- X[, comunes, drop = FALSE]
  X_alineada
}

formula_x <- stats::as.formula(
  paste("~", paste(variables_predictoras, collapse = " + "))
)
cat("Funciones de preprocesamiento preparadas.\n")

Funciones de preprocesamiento preparadas.


## 7. Preparar las matrices de clasificación

Se aplica el preprocesamiento ajustado con TRAIN a los tres periodos. La etiqueta objetivo no entra en la matriz de predictores.

In [ ]:
preparacion_clasificacion <- preparar_categorias_temporal(
  datos_train,
  datos_val,
  datos_test_final,
  variables_categoricas = variables_categoricas,
  frecuencia_minima = 2
)

train_clas <- preparacion_clasificacion$train
val_clas <- preparacion_clasificacion$validacion
test_clas <- preparacion_clasificacion$test

x_train <- crear_matriz_predictores(train_clas, formula_x)
columnas_x_clasificacion <- colnames(x_train)
x_val <- crear_matriz_predictores(val_clas, formula_x, columnas_x_clasificacion)
x_test <- crear_matriz_predictores(test_clas, formula_x, columnas_x_clasificacion)

y_train <- as.integer(train_clas$retraso_15_min)
y_val <- as.integer(val_clas$retraso_15_min)
y_test <- as.integer(test_clas$retraso_15_min)

stopifnot(
  identical(colnames(x_train), colnames(x_val)),
  identical(colnames(x_train), colnames(x_test)),
  nrow(x_train) == length(y_train),
  nrow(x_val) == length(y_val),
  nrow(x_test) == length(y_test)
)

cat("Dimensiones TRAIN:", nrow(x_train), "filas x", ncol(x_train), "variables\n")
cat("Dimensiones VALIDACIÓN:", nrow(x_val), "filas\n")
cat("Dimensiones TEST FINAL:", nrow(x_test), "filas\n")

Dimensiones TRAIN: 44495 filas x 457 variables
Dimensiones VALIDACIÓN: 14173 filas
Dimensiones TEST FINAL: 24739 filas


## 8. Entrenar el clasificador definitivo: Gradient Boosting

Se conserva Gradient Boosting como clasificador final de esta fase del TFM. Los parámetros quedan fijados antes de evaluar TEST FINAL. La selección del **umbral de decisión** se realizará en VALIDACIÓN, no en TEST.

In [ ]:
datos_gbm_train <- data.frame(
  retraso_15_min = y_train,
  as.data.frame(x_train),
  check.names = TRUE
)

datos_gbm_val <- as.data.frame(x_val)
datos_gbm_test <- as.data.frame(x_test)

set.seed(123)
modelo_gbm_final <- gbm::gbm(
  retraso_15_min ~ .,
  data = datos_gbm_train,
  distribution = "bernoulli",
  n.trees = 300,
  interaction.depth = 3,
  shrinkage = 0.03,
  n.minobsinnode = 20,
  bag.fraction = 0.5,
  train.fraction = 1,
  verbose = FALSE
)

prob_gbm_val <- predict(
  modelo_gbm_final,
  newdata = datos_gbm_val,
  n.trees = 300,
  type = "response"
)

cat("Gradient Boosting entrenado. Probabilidades de validación:", length(prob_gbm_val), "\n")

Gradient Boosting entrenado. Probabilidades de validación: 14173 


## 9. Elegir el umbral con VALIDACIÓN

El umbral se elige maximizando F1 en VALIDACIÓN. La tabla permite observar el equilibrio entre precisión y recall. Este paso no consulta las etiquetas del TEST FINAL.

In [ ]:
evaluar_umbral <- function(prob, real, umbral) {
  pred <- as.integer(prob >= umbral)
  tp <- sum(pred == 1 & real == 1)
  tn <- sum(pred == 0 & real == 0)
  fp <- sum(pred == 1 & real == 0)
  fn <- sum(pred == 0 & real == 1)

  precision <- if ((tp + fp) == 0) 0 else tp / (tp + fp)
  recall <- if ((tp + fn) == 0) 0 else tp / (tp + fn)
  especificidad <- if ((tn + fp) == 0) 0 else tn / (tn + fp)
  f1 <- if ((precision + recall) == 0) 0 else {
    2 * precision * recall / (precision + recall)
  }

  data.frame(
    umbral = umbral,
    precision = precision,
    recall = recall,
    especificidad = especificidad,
    F1 = f1
  )
}

umbrales <- seq(0.10, 0.60, by = 0.05)
resultados_validacion <- do.call(rbind, lapply(umbrales, function(u) {
  evaluar_umbral(prob_gbm_val, y_val, u)
}))

# En empates de F1, se prefiere mayor precisión.
resultados_validacion <- resultados_validacion[
  order(-resultados_validacion$F1, -resultados_validacion$precision),
]
row.names(resultados_validacion) <- NULL
umbral_gbm_definitivo <- resultados_validacion$umbral[1]

cat("Umbral elegido en VALIDACIÓN:", umbral_gbm_definitivo, "\n")
print(resultados_validacion, digits = 4, row.names = FALSE)

Umbral elegido en VALIDACIÓN: 0.1 
 umbral precision   recall especificidad       F1
   0.10   0.06322 0.483726        0.5192 0.111832
   0.15   0.04896 0.197531        0.7426 0.078466
   0.20   0.01742 0.016835        0.9363 0.017123
   0.25   0.05556 0.003367        0.9962 0.006349
   0.30   0.00000 0.000000        0.9990 0.000000
   0.35   0.00000 0.000000        1.0000 0.000000
   0.40   0.00000 0.000000        1.0000 0.000000
   0.45   0.00000 0.000000        1.0000 0.000000
   0.50   0.00000 0.000000        1.0000 0.000000
   0.55   0.00000 0.000000        1.0000 0.000000
   0.60   0.00000 0.000000        1.0000 0.000000


## 10. Evaluación final de clasificación en TEST

Se aplica el umbral seleccionado en VALIDACIÓN al tramo temporal más reciente y se calculan las métricas finales. No se vuelve a ajustar el umbral con el resultado del test.

In [ ]:
prob_gbm_test <- predict(
  modelo_gbm_final,
  newdata = datos_gbm_test,
  n.trees = 300,
  type = "response"
)

pred_gbm_test <- as.integer(prob_gbm_test >= umbral_gbm_definitivo)

TP <- sum(pred_gbm_test == 1 & y_test == 1)
TN <- sum(pred_gbm_test == 0 & y_test == 0)
FP <- sum(pred_gbm_test == 1 & y_test == 0)
FN <- sum(pred_gbm_test == 0 & y_test == 1)

precision_final <- if (TP + FP == 0) 0 else TP / (TP + FP)
recall_final <- if (TP + FN == 0) 0 else TP / (TP + FN)
especificidad_final <- if (TN + FP == 0) 0 else TN / (TN + FP)
f1_final <- if (precision_final + recall_final == 0) 0 else {
  2 * precision_final * recall_final / (precision_final + recall_final)
}

resultados_clasificacion_final <- data.frame(
  modelo = "Gradient Boosting",
  umbral = umbral_gbm_definitivo,
  filas = length(y_test),
  accuracy = (TP + TN) / length(y_test),
  precision = precision_final,
  recall = recall_final,
  especificidad = especificidad_final,
  F1 = f1_final,
  TP = TP,
  TN = TN,
  FP = FP,
  FN = FN
)

cat("Métricas finales de clasificación:\n")
print(resultados_clasificacion_final, digits = 4, row.names = FALSE)

cat("\nMatriz de confusión (filas = predicción; columnas = clase real):\n")
print(table(
  Prediccion = factor(pred_gbm_test, levels = c(0, 1)),
  Real = factor(y_test, levels = c(0, 1))
))

Métricas finales de clasificación:
            modelo umbral filas accuracy precision recall especificidad     F1
 Gradient Boosting    0.1 24739   0.4825      0.23 0.4855        0.4816 0.3122
   TP   TN   FP   FN
 2905 9032 9723 3079

Matriz de confusión (filas = predicción; columnas = clase real):
          Real
Prediccion    0    1
         0 9032 3079
         1 9723 2905


## 11. Preparar la regresión lineal

La regresión se entrena únicamente con vuelos cuya etiqueta es `RETRASO` (diferencia estimada de 15 minutos o más). Se conserva la regresión lineal como modelo de trabajo acordado en la comparación previa. El conjunto de VALIDACIÓN sirve para revisar el error antes de presentar el resultado de TEST FINAL.

In [ ]:
train_reg_original <- datos_train %>%
  filter(retraso_15_min == 1) %>%
  select(all_of(c(variables_predictoras, "retraso_minutos")))

val_reg_original <- datos_val %>%
  filter(retraso_15_min == 1) %>%
  select(all_of(c(variables_predictoras, "retraso_minutos")))

test_reg_original <- datos_test_final %>%
  filter(retraso_15_min == 1) %>%
  select(all_of(c("aena_airport_iata", "flight_number", "scheduled_local",
                  variables_predictoras, "retraso_minutos")))

if (nrow(train_reg_original) == 0 || nrow(val_reg_original) == 0 ||
    nrow(test_reg_original) == 0) {
  stop("No hay suficientes vuelos etiquetados como RETRASO para la regresión en uno de los periodos.")
}

# Se conservan identificadores del test aparte para poder exportar las predicciones.
ids_test_reg <- test_reg_original %>%
  select(any_of(c("aena_airport_iata", "flight_number", "scheduled_local")))

test_reg_features <- test_reg_original %>%
  select(all_of(c(variables_predictoras, "retraso_minutos")))

preparacion_regresion <- preparar_categorias_temporal(
  train_reg_original,
  val_reg_original,
  test_reg_features,
  variables_categoricas = variables_categoricas,
  frecuencia_minima = 2
)

train_reg <- preparacion_regresion$train
val_reg <- preparacion_regresion$validacion
test_reg <- preparacion_regresion$test

x_train_reg <- crear_matriz_predictores(train_reg, formula_x)
columnas_x_regresion <- colnames(x_train_reg)
x_val_reg <- crear_matriz_predictores(val_reg, formula_x, columnas_x_regresion)
x_test_reg <- crear_matriz_predictores(test_reg, formula_x, columnas_x_regresion)

cat("Vuelos retrasados en TRAIN:", nrow(train_reg), "\n")
cat("Vuelos retrasados en VALIDACIÓN:", nrow(val_reg), "\n")
cat("Vuelos retrasados en TEST FINAL:", nrow(test_reg), "\n")

Vuelos retrasados en TRAIN: 6191 
Vuelos retrasados en VALIDACIÓN: 891 
Vuelos retrasados en TEST FINAL: 5984 


## 12. Entrenar la regresión lineal y revisar VALIDACIÓN

La matriz de regresión se construye con las mismas variables predictoras. Se seleccionan columnas linealmente independientes mediante descomposición QR, para evitar coeficientes no estimables por redundancia exacta en las variables indicadoras.

In [ ]:
X_reg_train_completo <- cbind(Intercept = 1, as.matrix(x_train_reg))
qr_reg <- qr(X_reg_train_completo)

columnas_indice_reg <- qr_reg$pivot[seq_len(qr_reg$rank)]
columnas_independientes_reg <- colnames(X_reg_train_completo)[columnas_indice_reg]
X_reg_train <- X_reg_train_completo[, columnas_independientes_reg, drop = FALSE]

modelo_lm_regresion <- stats::lm.fit(
  x = X_reg_train,
  y = train_reg$retraso_minutos
)

predecir_regresion_lineal <- function(x) {
  X <- cbind(Intercept = 1, as.matrix(x))
  X <- X[, columnas_independientes_reg, drop = FALSE]
  as.numeric(X %*% modelo_lm_regresion$coefficients)
}

pred_reg_val <- predecir_regresion_lineal(x_val_reg)

calcular_metricas_regresion <- function(real, pred, conjunto) {
  validos <- is.finite(real) & is.finite(pred)
  if (!any(validos)) stop(paste("No hay predicciones válidas en", conjunto))
  data.frame(
    conjunto = conjunto,
    vuelos_evaluados = sum(validos),
    MAE_minutos = mean(abs(real[validos] - pred[validos])),
    RMSE_minutos = sqrt(mean((real[validos] - pred[validos])^2))
  )
}

metricas_regresion_validacion <- calcular_metricas_regresion(
  val_reg$retraso_minutos, pred_reg_val, "VALIDACIÓN"
)
cat("Métricas de regresión en VALIDACIÓN (diagnóstico):\n")
print(metricas_regresion_validacion, digits = 4, row.names = FALSE)

Métricas de regresión en VALIDACIÓN (diagnóstico):
   conjunto vuelos_evaluados MAE_minutos RMSE_minutos
 VALIDACIÓN              891       28.65        57.04


## 13. Evaluación final de regresión en TEST

El resultado se evalúa solo sobre los vuelos etiquetados como `RETRASO`. El MAE representa el error absoluto medio en minutos y el RMSE penaliza más los errores extremos. Los retrasos estimados extremos presentes en la fuente no se eliminan automáticamente; deben explicarse como limitación del conjunto.

In [ ]:
pred_reg_test <- predecir_regresion_lineal(x_test_reg)

metricas_regresion_test <- calcular_metricas_regresion(
  test_reg$retraso_minutos, pred_reg_test, "TEST FINAL"
)

cat("Métricas finales de regresión:\n")
print(metricas_regresion_test, digits = 4, row.names = FALSE)

predicciones_regresion_test <- data.frame(
  ids_test_reg,
  retraso_estimado_objetivo = test_reg$retraso_minutos,
  retraso_predicho = pred_reg_test,
  error_absoluto_minutos = abs(test_reg$retraso_minutos - pred_reg_test)
)

cat("\nDiez mayores errores absolutos de regresión:\n")
print(head(
  predicciones_regresion_test[order(
    predicciones_regresion_test$error_absoluto_minutos,
    decreasing = TRUE
  ), ],
  10
), row.names = FALSE)

Métricas finales de regresión:
   conjunto vuelos_evaluados MAE_minutos RMSE_minutos
 TEST FINAL             5984       40.18        70.35

Diez mayores errores absolutos de regresión:
 aena_airport_iata flight_number     scheduled_local retraso_estimado_objetivo
               BCN        FR9387 2026-10-17T21:20:00                      1440
               BCN        FR6985 2026-10-17T20:35:00                      1440
               BCN         FR774 2026-10-18T05:55:00                      1115
               BCN        FR4545 2026-10-18T06:20:00                      1094
               BCN        FR2918 2026-10-16T08:15:00                       833
               BCN        W46020 2026-10-17T21:45:00                       735
               MAD        I21545 2026-10-17T12:15:00                       704
               MAD        AT5429 2026-10-17T12:15:00                       704
               MAD        LA5405 2026-10-17T12:15:00                       704
               MAD       

## 14. Resumen de resultados y limitaciones

Al terminar la ejecución, conserva las tablas generadas y usa sus valores reales en la memoria. No copies métricas de ejecuciones previas, porque pueden corresponder a otra partición temporal.

En la interpretación final, deja explícito que:

- el objetivo se deriva de `estimated_local - scheduled_local`;
- el objetivo puede representar una estimación de demora en el instante de captura y no un retraso real observado;
- algunas fechas contienen muchas estimaciones iguales a las horas programadas porque los vuelos se capturaron antes de su ventana operativa;
- las métricas corresponden al periodo y al conjunto disponibles hasta el 19/10/2026;
- una predicción probabilística no garantiza el resultado individual de un vuelo.

In [ ]:
cat("=== CLASIFICACIÓN: TEST FINAL ===\n")
print(resultados_clasificacion_final, digits = 4, row.names = FALSE)

cat("\n=== REGRESIÓN: VALIDACIÓN ===\n")
print(metricas_regresion_validacion, digits = 4, row.names = FALSE)

cat("\n=== REGRESIÓN: TEST FINAL ===\n")
print(metricas_regresion_test, digits = 4, row.names = FALSE)

=== CLASIFICACIÓN: TEST FINAL ===
            modelo umbral filas accuracy precision recall especificidad     F1
 Gradient Boosting    0.1 24739   0.4825      0.23 0.4855        0.4816 0.3122
   TP   TN   FP   FN
 2905 9032 9723 3079

=== REGRESIÓN: VALIDACIÓN ===
   conjunto vuelos_evaluados MAE_minutos RMSE_minutos
 VALIDACIÓN              891       28.65        57.04

=== REGRESIÓN: TEST FINAL ===
   conjunto vuelos_evaluados MAE_minutos RMSE_minutos
 TEST FINAL             5984       40.18        70.35


## 15. Guardar resultados y modelos

Se guardan las métricas, las predicciones de TEST FINAL y un objeto RDS con los modelos y la información necesaria sobre el preprocesamiento. Los archivos quedan en el directorio de trabajo de Colab y pueden descargarse desde el panel de archivos.

In [ ]:
predicciones_clasificacion_test <- data.frame(
  scheduled_local = as.character(test_clas$scheduled_local),
  flight_number = test_clas$flight_number,
  aena_airport_iata = test_clas$aena_airport_iata,
  retraso_15_min_real = y_test,
  probabilidad_retraso = prob_gbm_test,
  prediccion_retraso_15_min = pred_gbm_test,
  umbral_aplicado = umbral_gbm_definitivo
)

write.csv(
  resultados_clasificacion_final,
  "aeropredict_metricas_clasificacion_test.csv",
  row.names = FALSE,
  fileEncoding = "UTF-8"
)
write.csv(
  resultados_validacion,
  "aeropredict_umbral_validacion.csv",
  row.names = FALSE,
  fileEncoding = "UTF-8"
)
write.csv(
  metricas_regresion_validacion,
  "aeropredict_metricas_regresion_validacion.csv",
  row.names = FALSE,
  fileEncoding = "UTF-8"
)
write.csv(
  metricas_regresion_test,
  "aeropredict_metricas_regresion_test.csv",
  row.names = FALSE,
  fileEncoding = "UTF-8"
)
write.csv(
  predicciones_clasificacion_test,
  "aeropredict_predicciones_clasificacion_test.csv",
  row.names = FALSE,
  fileEncoding = "UTF-8"
)
write.csv(
  predicciones_regresion_test,
  "aeropredict_predicciones_regresion_test.csv",
  row.names = FALSE,
  fileEncoding = "UTF-8"
)

artefacto_aeropredict <- list(
  fecha_corte_datos = as.character(fecha_corte_datos),
  variables_predictoras = variables_predictoras,
  variables_categoricas = variables_categoricas,
  formula_predictoras = formula_x,
  definicion_objetivo = "1 si retraso_minutos >= 15; 0 en caso contrario. El objetivo es estimated_local - scheduled_local, no retraso real observado.",
  cortes_temporales = list(
    inicio_validacion = as.character(corte_validacion),
    inicio_test_final = as.character(corte_test)
  ),
  clasificacion = list(
    modelo = modelo_gbm_final,
    umbral = umbral_gbm_definitivo,
    columnas_x = columnas_x_clasificacion,
    reglas_categorias = preparacion_clasificacion$reglas
  ),
  regresion = list(
    modelo = modelo_lm_regresion,
    columnas_x = columnas_independientes_reg,
    reglas_categorias = preparacion_regresion$reglas
  )
)

saveRDS(artefacto_aeropredict, "aeropredict_modelos_tfm.rds")
cat("Resultados CSV y artefacto RDS guardados en el directorio de trabajo.\n")

Resultados CSV y artefacto RDS guardados en el directorio de trabajo.


## Fin del notebook

Antes de entregar el TFM, ejecuta **Restart Kernel / Restart runtime** y luego **Run all** de arriba abajo. Comprueba que no quedan errores, conserva las tablas finales generadas por esa ejecución y evita compartir el notebook con salidas que contengan datos sensibles de conexión.

In [ ]:
# PASO 16 — Verificar la exportación de AeroPredict

# 1. Comprobar que el archivo existe
stopifnot(file.exists("aeropredict_modelos_tfm.rds"))

# 2. Cargar los modelos exportados
artefacto_cargado <- readRDS("aeropredict_modelos_tfm.rds")

stopifnot(
  !is.null(artefacto_cargado$clasificacion$modelo),
  !is.null(artefacto_cargado$regresion$modelo)
)

# 3. Verificar que la clasificación reproduce las probabilidades
prob_original <- predict(
  modelo_gbm_final,
  newdata = as.data.frame(x_test),
  n.trees = 300,
  type = "response"
)

prob_exportado <- predict(
  artefacto_cargado$clasificacion$modelo,
  newdata = as.data.frame(x_test),
  n.trees = 300,
  type = "response"
)

stopifnot(isTRUE(all.equal(
  as.numeric(prob_original),
  as.numeric(prob_exportado),
  tolerance = 1e-12
)))

# 4. Verificar que la regresión reproduce las predicciones
X_test_exportado <- cbind(Intercept = 1, as.matrix(x_test_reg))

X_test_exportado <- X_test_exportado[
  ,
  artefacto_cargado$regresion$columnas_x,
  drop = FALSE
]

pred_reg_exportado <- as.numeric(
  X_test_exportado %*%
    artefacto_cargado$regresion$modelo$coefficients
)

stopifnot(isTRUE(all.equal(
  as.numeric(pred_reg_test),
  pred_reg_exportado,
  tolerance = 1e-8
)))

cat("Correcto: los dos modelos se han exportado y recuperado.\n")
cat("Las predicciones originales y las recuperadas coinciden.\n")

Correcto: los dos modelos se han exportado y recuperado.
Las predicciones originales y las recuperadas coinciden.


# **Paso 1. Auditar los modelos actuales**

In [ ]:
cat("========== AUDITORÍA AEROPREDICT ==========\n\n")

# 1. Comprobar qué objetos tenemos en memoria
objetos <- c(
  "df_modelado",
  "datos_train",
  "datos_val",
  "datos_test_final",
  "modelo_gbm_final",
  "modelo_lm_regresion",
  "umbral_gbm_definitivo",
  "resultados_clasificacion_final",
  "metricas_regresion_validacion",
  "metricas_regresion_test",
  "artefacto_aeropredict"
)

estado <- data.frame(
  objeto = objetos,
  disponible = vapply(
    objetos,
    exists,
    logical(1),
    inherits = TRUE
  )
)

print(estado, row.names = FALSE)

# 2. Revisar el clasificador
if (exists("modelo_gbm_final")) {
  cat("\n--- CLASIFICADOR GBM ---\n")
  print(class(modelo_gbm_final))
  print(modelo_gbm_final[c(
    "n.trees",
    "interaction.depth",
    "shrinkage",
    "n.minobsinnode",
    "bag.fraction"
  )])
}

# 3. Consultar el umbral elegido
if (exists("umbral_gbm_definitivo")) {
  cat("\n--- UMBRAL ELEGIDO EN VALIDACIÓN ---\n")
  print(umbral_gbm_definitivo)
}

# 4. Revisar tamaños de las particiones
cat("\n--- PARTICIONES ---\n")

for (nombre in c(
  "datos_train",
  "datos_val",
  "datos_test_final",
  "train_reg",
  "val_reg",
  "test_reg"
)) {
  if (exists(nombre)) {
    objeto <- get(nombre)
    cat(nombre, ":", nrow(objeto), "filas\n")
  }
}

# 5. Revisar la regresión y sus coeficientes
if (exists("modelo_lm_regresion")) {
  cat("\n--- REGRESIÓN LINEAL ---\n")
  print(class(modelo_lm_regresion))
  print(modelo_lm_regresion$coefficients)
}

# 6. Mostrar las métricas que estén disponibles
for (nombre in c(
  "resultados_clasificacion_final",
  "metricas_regresion_validacion",
  "metricas_regresion_test"
)) {
  if (exists(nombre)) {
    cat("\n---", nombre, "---\n")
    print(get(nombre))
  }
}

cat("\n========== FIN DE AUDITORÍA ==========\n")

========== AUDITORÍA AEROPREDICT ==========

                         objeto disponible
                    df_modelado       TRUE
                    datos_train       TRUE
                      datos_val       TRUE
               datos_test_final       TRUE
               modelo_gbm_final       TRUE
            modelo_lm_regresion       TRUE
          umbral_gbm_definitivo       TRUE
 resultados_clasificacion_final       TRUE
  metricas_regresion_validacion       TRUE
        metricas_regresion_test       TRUE
          artefacto_aeropredict       TRUE

--- CLASIFICADOR GBM ---
[1] "gbm"
$n.trees
[1] 300

$interaction.depth
[1] 3

$shrinkage
[1] 0.03

$n.minobsinnode
[1] 20

$bag.fraction
[1] 0.5


--- UMBRAL ELEGIDO EN VALIDACIÓN ---
[1] 0.1

--- PARTICIONES ---
datos_train : 44495 filas
datos_val : 14173 filas
datos_test_final : 24739 filas
train_reg : 6191 filas
val_reg : 891 filas
test_reg : 5984 filas

--- REGRESIÓN LINEAL ---
[1] "list"
             Intercept            hora_vu

# **3. Siguiente paso: diagnosticar antes de reentrenar**

In [ ]:
cat("========== DIAGNÓSTICO AEROPREDICT ==========\n")

# 1. Distribución de retrasos por partición
for (nombre in c("datos_train", "datos_val", "datos_test_final")) {
  if (exists(nombre)) {
    d <- get(nombre)

    cat("\n---", nombre, "---\n")
    print(table(d$retraso_15_min))
    print(round(prop.table(table(d$retraso_15_min)), 4))
  }
}

# 2. Umbrales evaluados en validación
if (exists("resultados_validacion")) {
  cat("\n--- MEJORES UMBRALES EN VALIDACIÓN ---\n")

  tabla <- resultados_validacion[
    order(-resultados_validacion$F1,
          -resultados_validacion$precision),
  ]

  print(head(tabla, 10), row.names = FALSE)
}

# 3. Distribución del objetivo de regresión
for (nombre in c("train_reg", "val_reg", "test_reg")) {
  if (exists(nombre)) {
    d <- get(nombre)

    cat("\n---", nombre, ": retraso_minutos ---\n")
    print(summary(d$retraso_minutos))

    print(quantile(
      d$retraso_minutos,
      probs = c(0, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99, 1),
      na.rm = TRUE
    ))
  }
}

# 4. Comparar la regresión con una referencia sencilla:
# predecir siempre la mediana de TRAIN.
if (exists("train_reg") && exists("test_reg") &&
    exists("pred_reg_test")) {

  mediana_train <- median(
    train_reg$retraso_minutos,
    na.rm = TRUE
  )

  real <- test_reg$retraso_minutos
  pred_lineal <- as.numeric(pred_reg_test)

  validos <- is.finite(real) & is.finite(pred_lineal)
  real <- real[validos]
  pred_lineal <- pred_lineal[validos]

  mae_lineal <- mean(abs(real - pred_lineal))
  mae_referencia <- mean(abs(real - mediana_train))

  cat("\n--- COMPARACIÓN DE REGRESIÓN ---\n")
  cat("MAE modelo lineal:", round(mae_lineal, 3), "\n")
  cat("MAE prediciendo siempre la mediana:",
      round(mae_referencia, 3), "\n")
  cat("Mediana de TRAIN:", round(mediana_train, 3), "\n")
}

cat("\n========== FIN DEL DIAGNÓSTICO ==========\n")

========== DIAGNÓSTICO AEROPREDICT ==========

--- datos_train ---

    0     1 
38304  6191 

     0      1 
0.8609 0.1391 

--- datos_val ---

    0     1 
13282   891 

     0      1 
0.9371 0.0629 

--- datos_test_final ---

    0     1 
18755  5984 

     0      1 
0.7581 0.2419 

--- MEJORES UMBRALES EN VALIDACIÓN ---
 umbral  precision      recall especificidad          F1
   0.10 0.06322429 0.483726150     0.5191989 0.111831863
   0.15 0.04895688 0.197530864     0.7425839 0.078466340
   0.20 0.01742160 0.016835017     0.9363048 0.017123288
   0.25 0.05555556 0.003367003     0.9961602 0.006349206
   0.30 0.00000000 0.000000000     0.9990212 0.000000000
   0.35 0.00000000 0.000000000     1.0000000 0.000000000
   0.40 0.00000000 0.000000000     1.0000000 0.000000000
   0.45 0.00000000 0.000000000     1.0000000 0.000000000
   0.50 0.00000000 0.000000000     1.0000000 0.000000000
   0.55 0.00000000 0.000000000     1.0000000 0.000000000

--- train_reg : retraso_minutos ---
   Min. 1s

# **2. Siguiente paso: revisar las etiquetas y los valores extremos**

In [ ]:
cat("========== RETRASOS POR FECHA ==========\n")

# Porcentaje de vuelos etiquetados como retrasados cada día
diagnostico_diario <- df_modelado %>%
  group_by(fecha_vuelo) %>%
  summarise(
    vuelos = n(),
    retrasos_15 = sum(retraso_15_min == 1),
    porcentaje_retrasos = mean(retraso_15_min == 1) * 100,
    mediana_minutos = median(retraso_minutos),
    p95_minutos = as.numeric(
      quantile(retraso_minutos, 0.95, na.rm = TRUE)
    ),
    max_minutos = max(retraso_minutos, na.rm = TRUE),
    .groups = "drop"
  ) %>%
  arrange(fecha_vuelo)

cat("\n--- Primeras fechas ---\n")
print(head(diagnostico_diario, 10), row.names = FALSE)

cat("\n--- Últimas fechas ---\n")
print(tail(diagnostico_diario, 20), row.names = FALSE)

cat("\n--- Días con mayor porcentaje de retrasos (mínimo 50 vuelos) ---\n")
print(
  diagnostico_diario %>%
    filter(vuelos >= 50) %>%
    arrange(desc(porcentaje_retrasos)) %>%
    head(15),
  row.names = FALSE
)

cat("\n========== CASOS EXTREMOS ==========\n")

# Examinar los mayores retrasos estimados
columnas_revision <- c(
  "fecha_vuelo",
  "aena_airport_iata",
  "flight_number",
  "flight_type",
  "scheduled_local",
  "hora_vuelo",
  "dia_semana",
  "airline_iata",
  "other_airport_iata",
  "retraso_minutos",
  "retraso_15_min"
)

casos_extremos <- df_modelado %>%
  filter(retraso_minutos >= 180) %>%
  arrange(desc(retraso_minutos)) %>%
  select(any_of(columnas_revision)) %>%
  head(20)

print(casos_extremos, row.names = FALSE)

cat("\n========== FIN DEL ANÁLISIS ==========\n")

========== RETRASOS POR FECHA ==========

--- Primeras fechas ---
# A tibble: 10 × 7
   fecha_vuelo vuelos retrasos_15 porcentaje_retrasos mediana_minutos
   <date>       <int>       <int>               <dbl>           <dbl>
 1 2026-08-01     860         588              68.4              32.5
 2 2026-08-02    3428        1107              32.3               8  
 3 2026-08-03    3512        1089              31.0               4  
 4 2026-08-04    3430          39               1.14              0  
 5 2026-08-05    3448         912              26.5               3  
 6 2026-08-06    3442          54               1.57              0  
 7 2026-08-07    3449           0               0                 0  
 8 2026-08-08      36           0               0                 0  
 9 2026-08-28    3381         442              13.1               0  
10 2026-08-29    3271           7               0.214             0  
# ℹ 2 more variables: p95_minutos <dbl>, max_minutos <dbl>

--- Últimas fec

In [ ]:
# ============================================================
# AEROPREDICT — COMPARACIÓN INICIAL DE CLASIFICADORES
# Selección mediante VALIDACIÓN; no utiliza TEST para elegir.
# ============================================================

# Instalar los paquetes necesarios si faltan
paquetes_nuevos <- c("rpart", "ranger")
faltantes <- setdiff(
  paquetes_nuevos,
  rownames(installed.packages())
)

if (length(faltantes) > 0) {
  install.packages(
    faltantes,
    repos = "https://cloud.r-project.org"
  )
}

# Preparar las matrices como data.frames
datos_clas_train_bench <- data.frame(
  clase = factor(
    ifelse(y_train == 1, "RETRASO", "NO_RETRASO"),
    levels = c("NO_RETRASO", "RETRASO")
  ),
  as.data.frame(x_train),
  check.names = TRUE
)

datos_clas_val_bench <- as.data.frame(x_val)
names(datos_clas_val_bench) <- names(datos_clas_train_bench)[-1]

# 1. Árbol de decisión
set.seed(123)

modelo_arbol_bench <- rpart::rpart(
  clase ~ .,
  data = datos_clas_train_bench,
  method = "class",
  control = rpart::rpart.control(
    cp = 0.001,
    minsplit = 50,
    maxdepth = 10,
    xval = 0
  )
)

prob_arbol_val <- predict(
  modelo_arbol_bench,
  newdata = datos_clas_val_bench,
  type = "prob"
)[, "RETRASO"]

# 2. Random Forest
set.seed(123)

modelo_rf_bench <- ranger::ranger(
  clase ~ .,
  data = datos_clas_train_bench,
  probability = TRUE,
  num.trees = 300,
  mtry = max(1, floor(sqrt(ncol(x_train)))),
  min.node.size = 10,
  importance = "none",
  seed = 123,
  num.threads = 2
)

prob_rf_val <- predict(
  modelo_rf_bench,
  data = datos_clas_val_bench
)$predictions[, "RETRASO"]

# 3. Evaluación de umbrales en VALIDACIÓN
# Para comparar los modelos bajo el mismo criterio,
# elegimos el umbral con mejor F1 en cada candidato.

calcular_auc <- function(prob, real) {
  positivos <- sum(real == 1)
  negativos <- sum(real == 0)

  if (positivos == 0 || negativos == 0) return(NA_real_)

  rangos <- rank(prob, ties.method = "average")

  (sum(rangos[real == 1]) -
     positivos * (positivos + 1) / 2) /
    (positivos * negativos)
}

evaluar_candidato <- function(nombre, prob, real) {
  stopifnot(
    length(prob) == length(real),
    all(is.finite(prob))
  )

  umbrales <- sort(unique(c(
    0.001,
    seq(0.01, 0.60, by = 0.01),
    0.70, 0.80, 0.90, 0.95, 0.99
  )))

  tabla <- do.call(rbind, lapply(umbrales, function(u) {
    pred <- as.integer(prob >= u)

    tp <- sum(pred == 1 & real == 1)
    tn <- sum(pred == 0 & real == 0)
    fp <- sum(pred == 1 & real == 0)
    fn <- sum(pred == 0 & real == 1)

    precision <- if (tp + fp == 0) 0 else tp / (tp + fp)
    recall <- if (tp + fn == 0) 0 else tp / (tp + fn)
    especificidad <- if (tn + fp == 0) 0 else tn / (tn + fp)
    f1 <- if (precision + recall == 0) {
      0
    } else {
      2 * precision * recall / (precision + recall)
    }

    data.frame(
      modelo = nombre,
      umbral = u,
      precision = precision,
      recall = recall,
      especificidad = especificidad,
      F1 = f1,
      porcentaje_predicho_retraso = mean(pred == 1) * 100
    )
  }))

  tabla$AUC <- calcular_auc(prob, real)
  tabla
}

# Incorporamos el GBM existente y los dos candidatos nuevos
comparacion_umbral <- rbind(
  evaluar_candidato(
    "Gradient Boosting actual",
    prob_gbm_val, y_val
  ),
  evaluar_candidato(
    "Árbol de decisión",
    prob_arbol_val, y_val
  ),
  evaluar_candidato(
    "Random Forest",
    prob_rf_val, y_val
  )
)

# Mejor umbral por F1; en caso de empate, mayor precisión
indices_mejores <- unlist(lapply(
  split(seq_len(nrow(comparacion_umbral)),
        comparacion_umbral$modelo),
  function(idx) {
    orden <- order(
      -comparacion_umbral$F1[idx],
      -comparacion_umbral$precision[idx]
    )
    idx[orden[1]]
  }
))

mejores_clasificadores <- comparacion_umbral[
  indices_mejores,
  c(
    "modelo", "umbral", "precision", "recall",
    "especificidad", "F1",
    "porcentaje_predicho_retraso", "AUC"
  )
]

mejores_clasificadores <- mejores_clasificadores[
  order(-mejores_clasificadores$F1),
]

cat("\n===== MEJORES RESULTADOS EN VALIDACIÓN =====\n")
print(mejores_clasificadores, digits = 4, row.names = FALSE)

cat("\n===== MEJORES UMBRALES POR MODELO =====\n")
print(
  head(
    comparacion_umbral[
      order(-comparacion_umbral$F1,
            -comparacion_umbral$precision),
    ],
    15
  ),
  digits = 4,
  row.names = FALSE
)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependency ‘RcppEigen’




Growing trees.. Progress: 16%. Estimated remaining time: 2 minutes, 46 seconds.
Growing trees.. Progress: 32%. Estimated remaining time: 2 minutes, 11 seconds.
Growing trees.. Progress: 49%. Estimated remaining time: 1 minute, 38 seconds.
Growing trees.. Progress: 66%. Estimated remaining time: 1 minute, 5 seconds.
Growing trees.. Progress: 86%. Estimated remaining time: 26 seconds.

===== MEJORES RESULTADOS EN VALIDACIÓN =====
                   modelo umbral precision recall especificidad     F1
 Gradient Boosting actual   0.08   0.08175 0.8956       0.32510 0.1498
            Random Forest   0.03   0.06502 0.9203       0.11226 0.1215
        Árbol de decisión   0.03   0.06444 0.9293       0.09487 0.1205
 porcentaje_predicho_retraso    AUC
                       68.88 0.5514
                       88.98 0.5075
                       90.67 0.4040

===== MEJORES UMBRALES POR MODELO =====
                   modelo umbral precision recall especificidad     F1
 Gradient Boosting actual  0

In [ ]:
# ============================================================
# AEROPREDICT — COMPARACIÓN DE MODELOS DE REGRESIÓN
# Selección con VALIDACIÓN, sin consultar TEST FINAL.
# ============================================================

# Comprobar que existen las matrices y los datos necesarios
stopifnot(
  exists("x_train_reg"),
  exists("x_val_reg"),
  exists("train_reg"),
  exists("val_reg"),
  exists("predecir_regresion_lineal")
)

# Preparar los mismos predictores para todos los modelos
datos_reg_train_bench <- data.frame(
  retraso_minutos = as.numeric(train_reg$retraso_minutos),
  as.data.frame(x_train_reg),
  check.names = TRUE
)

datos_reg_val_bench <- as.data.frame(x_val_reg)
names(datos_reg_val_bench) <- names(datos_reg_train_bench)[-1]

y_val_reg_bench <- as.numeric(val_reg$retraso_minutos)

stopifnot(
  nrow(datos_reg_train_bench) == nrow(x_train_reg),
  nrow(datos_reg_val_bench) == length(y_val_reg_bench),
  identical(
    names(datos_reg_train_bench)[-1],
    names(datos_reg_val_bench)
  )
)

# 1. Referencias sencillas
mediana_train_reg <- median(
  datos_reg_train_bench$retraso_minutos
)

media_train_reg <- mean(
  datos_reg_train_bench$retraso_minutos
)

pred_mediana_val <- rep(
  mediana_train_reg,
  length(y_val_reg_bench)
)

pred_media_val <- rep(
  media_train_reg,
  length(y_val_reg_bench)
)

# 2. Regresión lineal actual
pred_lineal_val <- predecir_regresion_lineal(x_val_reg)

# 3. Gradient Boosting para regresión
set.seed(123)

modelo_gbm_reg_bench <- gbm::gbm(
  retraso_minutos ~ .,
  data = datos_reg_train_bench,
  distribution = "gaussian",
  n.trees = 300,
  interaction.depth = 3,
  shrinkage = 0.03,
  n.minobsinnode = 20,
  bag.fraction = 0.5,
  train.fraction = 1,
  verbose = FALSE
)

pred_gbm_reg_val <- predict(
  modelo_gbm_reg_bench,
  newdata = datos_reg_val_bench,
  n.trees = 300,
  type = "response"
)

# 4. Random Forest para regresión
set.seed(123)

modelo_rf_reg_bench <- ranger::ranger(
  retraso_minutos ~ .,
  data = datos_reg_train_bench,
  num.trees = 300,
  min.node.size = 10,
  seed = 123,
  num.threads = 2
)

pred_rf_reg_val <- predict(
  modelo_rf_reg_bench,
  data = datos_reg_val_bench
)$predictions

# 5. Comparar métricas en VALIDACIÓN
calcular_metricas_reg_bench <- function(real, pred, modelo) {
  validos <- is.finite(real) & is.finite(pred)

  data.frame(
    modelo = modelo,
    vuelos = sum(validos),
    MAE = mean(abs(real[validos] - pred[validos])),
    RMSE = sqrt(mean((real[validos] - pred[validos])^2))
  )
}

comparacion_regresion <- do.call(
  rbind,
  list(
    calcular_metricas_reg_bench(
      y_val_reg_bench, pred_mediana_val,
      "Referencia: mediana de TRAIN"
    ),
    calcular_metricas_reg_bench(
      y_val_reg_bench, pred_media_val,
      "Referencia: media de TRAIN"
    ),
    calcular_metricas_reg_bench(
      y_val_reg_bench, pred_lineal_val,
      "Regresión lineal actual"
    ),
    calcular_metricas_reg_bench(
      y_val_reg_bench, pred_gbm_reg_val,
      "Gradient Boosting regresión"
    ),
    calcular_metricas_reg_bench(
      y_val_reg_bench, pred_rf_reg_val,
      "Random Forest regresión"
    )
  )
)

comparacion_regresion <- comparacion_regresion[
  order(comparacion_regresion$MAE),
]

cat("\n===== COMPARACIÓN EN VALIDACIÓN =====\n")
print(comparacion_regresion, digits = 4, row.names = FALSE)


===== COMPARACIÓN EN VALIDACIÓN =====
                       modelo vuelos   MAE  RMSE
 Referencia: mediana de TRAIN    891 24.54 57.39
      Random Forest regresión    891 27.17 55.79
  Gradient Boosting regresión    891 27.27 55.94
      Regresión lineal actual    891 28.65 57.04
   Referencia: media de TRAIN    891 29.17 56.01


# **Paso 3. Validación temporal repetida de la regresión**

In [ ]:
# ============================================================
# AEROPREDICT — VALIDACIÓN TEMPORAL REPETIDA DE REGRESIÓN
# 3 ventanas temporales x 4 candidatos
# No utiliza datos de TEST FINAL original.
# ============================================================

library(dplyr)
library(gbm)
library(ranger)

set.seed(123)

# 1. Crear el conjunto de desarrollo sin incorporar TEST FINAL
datos_desarrollo <- bind_rows(datos_train, datos_val) %>%
  arrange(fecha_vuelo)

fechas_desarrollo <- sort(unique(datos_desarrollo$fecha_vuelo))
n_fechas <- length(fechas_desarrollo)

if (n_fechas < 15) {
  stop("No hay suficientes fechas para realizar esta validación temporal.")
}

# Cada fila contiene la fracción de fechas usada para:
# entrenamiento acumulativo y fin de la ventana de validación.
ventanas <- data.frame(
  fold = 1:3,
  fin_train = c(0.40, 0.60, 0.80),
  fin_val = c(0.60, 0.80, 1.00)
)

resultados_repetidos <- list()
k <- 0L

for (i in seq_len(nrow(ventanas))) {

  fin_train <- max(
    2L,
    floor(n_fechas * ventanas$fin_train[i])
  )

  fin_val <- min(
    n_fechas,
    floor(n_fechas * ventanas$fin_val[i])
  )

  if (fin_val <= fin_train) {
    next
  }

  fechas_train <- fechas_desarrollo[seq_len(fin_train)]
  fechas_val <- fechas_desarrollo[
    seq.int(fin_train + 1L, fin_val)
  ]

  train_fold <- datos_desarrollo %>%
    filter(fecha_vuelo %in% fechas_train)

  val_fold <- datos_desarrollo %>%
    filter(fecha_vuelo %in% fechas_val)

  # La regresión solo aprende de vuelos etiquetados como retrasados.
  train_reg_fold <- train_fold %>%
    filter(retraso_15_min == 1) %>%
    select(all_of(c(variables_predictoras, "retraso_minutos")))

  val_reg_fold <- val_fold %>%
    filter(retraso_15_min == 1) %>%
    select(all_of(c(variables_predictoras, "retraso_minutos")))

  if (nrow(train_reg_fold) < 30 || nrow(val_reg_fold) < 10) {
    cat(
      "Se omite fold", ventanas$fold[i],
      ": no hay suficientes vuelos retrasados.\n"
    )
    next
  }

  # Aprender las reglas categóricas únicamente con el TRAIN del fold.
  prep <- preparar_categorias_temporal(
    train = train_reg_fold,
    validacion = val_reg_fold,
    test = val_reg_fold,
    variables_categoricas = variables_categoricas,
    frecuencia_minima = 2
  )

  xtr <- crear_matriz_predictores(
    prep$train, formula_x
  )

  columnas_fold <- colnames(xtr)

  xva <- crear_matriz_predictores(
    prep$validacion, formula_x, columnas_fold
  )

  ytr <- as.numeric(prep$train$retraso_minutos)
  yva <- as.numeric(prep$validacion$retraso_minutos)

  stopifnot(
    nrow(xtr) == length(ytr),
    nrow(xva) == length(yva),
    all(is.finite(ytr)),
    all(is.finite(yva))
  )

  # Referencia: mediana del TRAIN de este fold
  pred_mediana <- rep(median(ytr), length(yva))

  # Regresión lineal: construir matrices y eliminar dependencias exactas
  Xtr_completo <- cbind(Intercept = 1, as.matrix(xtr))
  Xva_completo <- cbind(Intercept = 1, as.matrix(xva))

  qr_fold <- qr(Xtr_completo)
  indices <- qr_fold$pivot[seq_len(qr_fold$rank)]
  columnas_lm <- colnames(Xtr_completo)[indices]

  lm_fold <- stats::lm.fit(
    x = Xtr_completo[, columnas_lm, drop = FALSE],
    y = ytr
  )

  pred_lm <- as.numeric(
    Xva_completo[, columnas_lm, drop = FALSE] %*%
      lm_fold$coefficients
  )

  # Datos comunes para GBM y Random Forest
  datos_tr_modelo <- data.frame(
    retraso_minutos = ytr,
    as.data.frame(xtr),
    check.names = TRUE
  )

  datos_va_modelo <- as.data.frame(xva)
  names(datos_va_modelo) <- names(datos_tr_modelo)[-1]

  # Gradient Boosting
  set.seed(1200 + i)

  gbm_fold <- gbm::gbm(
    retraso_minutos ~ .,
    data = datos_tr_modelo,
    distribution = "gaussian",
    n.trees = 300,
    interaction.depth = 3,
    shrinkage = 0.03,
    n.minobsinnode = 20,
    bag.fraction = 0.5,
    train.fraction = 1,
    verbose = FALSE
  )

  pred_gbm <- as.numeric(predict(
    gbm_fold,
    newdata = datos_va_modelo,
    n.trees = 300,
    type = "response"
  ))

  # Random Forest
  set.seed(2200 + i)

  rf_fold <- ranger::ranger(
    retraso_minutos ~ .,
    data = datos_tr_modelo,
    num.trees = 300,
    min.node.size = 10,
    seed = 2200 + i,
    num.threads = 2
  )

  pred_rf <- as.numeric(predict(
    rf_fold,
    data = datos_va_modelo
  )$predictions)

  # Medir el error en la ventana de validación
  medir <- function(nombre, pred) {
    validos <- is.finite(yva) & is.finite(pred)

    data.frame(
      fold = ventanas$fold[i],
      modelo = nombre,
      fechas_train = length(fechas_train),
      vuelos_train = length(ytr),
      fechas_validacion = length(fechas_val),
      vuelos_validacion = sum(validos),
      MAE = mean(abs(yva[validos] - pred[validos])),
      RMSE = sqrt(mean((yva[validos] - pred[validos])^2))
    )
  }

  k <- k + 1L

  resultados_repetidos[[k]] <- do.call(
    rbind,
    list(
      medir("Referencia: mediana", pred_mediana),
      medir("Regresión lineal", pred_lm),
      medir("Gradient Boosting", pred_gbm),
      medir("Random Forest", pred_rf)
    )
  )

  cat(
    "Fold", ventanas$fold[i], "completado |",
    "train:", length(ytr), "vuelos retrasados |",
    "validación:", length(yva), "vuelos retrasados\n"
  )
}

if (length(resultados_repetidos) == 0) {
  stop(
    "No se completó ningún fold. Revisa las fechas y las categorías no vistas."
  )
}

tabla_folds <- bind_rows(resultados_repetidos)

cat("\n===== MÉTRICAS POR VENTANA TEMPORAL =====\n")
print(
  tabla_folds %>% arrange(fold, MAE),
  digits = 4,
  row.names = FALSE
)

cat("\n===== RESUMEN ENTRE VENTANAS =====\n")

resumen_repetido <- tabla_folds %>%
  group_by(modelo) %>%
  summarise(
    folds = n(),
    MAE_medio = mean(MAE),
    MAE_sd = sd(MAE),
    RMSE_medio = mean(RMSE),
    RMSE_sd = sd(RMSE),
    .groups = "drop"
  ) %>%
  arrange(MAE_medio)

print(resumen_repetido, digits = 4, row.names = FALSE)

Warning message in gbm.fit(x = x, y = y, offset = offset, distribution = distribution, :
“variable 3: hora_vuelo3 has no variation.”


Fold 1 completado | train: 4231 vuelos retrasados | validación: 981 vuelos retrasados


Warning message in gbm.fit(x = x, y = y, offset = offset, distribution = distribution, :
“variable 3: hora_vuelo3 has no variation.”


Fold 2 completado | train: 5212 vuelos retrasados | validación: 991 vuelos retrasados
Fold 3 completado | train: 6203 vuelos retrasados | validación: 879 vuelos retrasados

===== MÉTRICAS POR VENTANA TEMPORAL =====
 fold              modelo fechas_train vuelos_train fechas_validacion
    1 Referencia: mediana            9         4231                 4
    1    Regresión lineal            9         4231                 4
    1       Random Forest            9         4231                 4
    1   Gradient Boosting            9         4231                 4
    2 Referencia: mediana           13         5212                 5
    2   Gradient Boosting           13         5212                 5
    2       Random Forest           13         5212                 5
    2    Regresión lineal           13         5212                 5
    3 Referencia: mediana           18         6203                 5
    3   Gradient Boosting           18         6203                 5
    3       Ran

# **2. Siguiente experimento: regresión con transformación logarítmica**

In [ ]:
# ============================================================
# AEROPREDICT — EXPERIMENTO CON OBJETIVO LOGARÍTMICO
# Usa TRAIN/VALIDACIÓN de desarrollo, nunca TEST FINAL.
# ============================================================

library(dplyr)
library(gbm)
library(ranger)

resultados_log <- list()
k_log <- 0L

for (i in seq_len(nrow(ventanas))) {

  fin_train <- max(2L, floor(n_fechas * ventanas$fin_train[i]))
  fin_val <- min(n_fechas, floor(n_fechas * ventanas$fin_val[i]))

  if (fin_val <= fin_train) next

  fechas_train <- fechas_desarrollo[seq_len(fin_train)]
  fechas_val <- fechas_desarrollo[seq.int(fin_train + 1L, fin_val)]

  train_fold <- datos_desarrollo %>%
    filter(fecha_vuelo %in% fechas_train)

  val_fold <- datos_desarrollo %>%
    filter(fecha_vuelo %in% fechas_val)

  train_reg_fold <- train_fold %>%
    filter(retraso_15_min == 1) %>%
    select(all_of(c(variables_predictoras, "retraso_minutos")))

  val_reg_fold <- val_fold %>%
    filter(retraso_15_min == 1) %>%
    select(all_of(c(variables_predictoras, "retraso_minutos")))

  if (nrow(train_reg_fold) < 30 || nrow(val_reg_fold) < 10) next

  prep <- preparar_categorias_temporal(
    train = train_reg_fold,
    validacion = val_reg_fold,
    test = val_reg_fold,
    variables_categoricas = variables_categoricas,
    frecuencia_minima = 2
  )

  xtr0 <- crear_matriz_predictores(prep$train, formula_x)

  # Eliminar columnas constantes según TRAIN de esta ventana.
  variables_no_constantes <- vapply(
    seq_len(ncol(xtr0)),
    function(j) length(unique(xtr0[, j])) > 1,
    logical(1)
  )

  columnas_fold <- colnames(xtr0)[variables_no_constantes]
  xtr <- xtr0[, variables_no_constantes, drop = FALSE]

  xva <- crear_matriz_predictores(
    prep$validacion,
    formula_x,
    columnas = columnas_fold
  )

  ytr <- as.numeric(prep$train$retraso_minutos)
  yva <- as.numeric(prep$validacion$retraso_minutos)

  # El objetivo de todos los candidatos logarítmicos es log1p(minutos).
  ytr_log <- log1p(ytr)

  # Regresión lineal sobre log1p.
  Xtr_completo <- cbind(Intercept = 1, as.matrix(xtr))
  Xva_completo <- cbind(Intercept = 1, as.matrix(xva))

  qr_fold <- qr(Xtr_completo)
  idx <- qr_fold$pivot[seq_len(qr_fold$rank)]
  columnas_lm <- colnames(Xtr_completo)[idx]

  lm_log <- stats::lm.fit(
    x = Xtr_completo[, columnas_lm, drop = FALSE],
    y = ytr_log
  )

  pred_lm_log <- pmax(
    15,
    expm1(as.numeric(
      Xva_completo[, columnas_lm, drop = FALSE] %*%
        lm_log$coefficients
    ))
  )

  # Preparar datos para los modelos de árboles.
  datos_tr <- data.frame(
    log_retraso = ytr_log,
    as.data.frame(xtr),
    check.names = TRUE
  )

  datos_va <- as.data.frame(xva)
  names(datos_va) <- names(datos_tr)[-1]

  # Gradient Boosting con objetivo transformado.
  set.seed(3200 + i)

  gbm_log <- gbm::gbm(
    log_retraso ~ .,
    data = datos_tr,
    distribution = "gaussian",
    n.trees = 300,
    interaction.depth = 3,
    shrinkage = 0.03,
    n.minobsinnode = 20,
    bag.fraction = 0.5,
    train.fraction = 1,
    verbose = FALSE
  )

  pred_gbm_log <- pmax(
    15,
    expm1(as.numeric(predict(
      gbm_log,
      newdata = datos_va,
      n.trees = 300,
      type = "response"
    )))
  )

  # Random Forest con objetivo transformado.
  set.seed(4200 + i)

  rf_log <- ranger::ranger(
    log_retraso ~ .,
    data = datos_tr,
    num.trees = 300,
    min.node.size = 10,
    seed = 4200 + i,
    num.threads = 2
  )

  pred_rf_log <- pmax(
    15,
    expm1(as.numeric(predict(
      rf_log,
      data = datos_va
    )$predictions))
  )

  medir_log <- function(nombre, pred) {
    validos <- is.finite(yva) & is.finite(pred)

    data.frame(
      fold = ventanas$fold[i],
      modelo = nombre,
      vuelos_validacion = sum(validos),
      MAE = mean(abs(yva[validos] - pred[validos])),
      RMSE = sqrt(mean((yva[validos] - pred[validos])^2))
    )
  }

  k_log <- k_log + 1L
  resultados_log[[k_log]] <- rbind(
    medir_log("Regresión lineal log1p", pred_lm_log),
    medir_log("Gradient Boosting log1p", pred_gbm_log),
    medir_log("Random Forest log1p", pred_rf_log)
  )

  cat("Fold", ventanas$fold[i], "completado.\n")
}

if (length(resultados_log) == 0) {
  stop("No se completó ningún fold del experimento logarítmico.")
}

tabla_log <- bind_rows(resultados_log)

resumen_log <- tabla_log %>%
  group_by(modelo) %>%
  summarise(
    folds = n(),
    MAE_medio = mean(MAE),
    MAE_sd = sd(MAE),
    RMSE_medio = mean(RMSE),
    RMSE_sd = sd(RMSE),
    .groups = "drop"
  )

# Comparar con las ejecuciones sobre la escala original.
resumen_original <- tabla_folds %>%
  group_by(modelo) %>%
  summarise(
    folds = n(),
    MAE_medio = mean(MAE),
    MAE_sd = sd(MAE),
    RMSE_medio = mean(RMSE),
    RMSE_sd = sd(RMSE),
    .groups = "drop"
  )

cat("\n===== RESULTADOS LOGARÍTMICOS POR VENTANA =====\n")
print(tabla_log, digits = 4, row.names = FALSE)

cat("\n===== COMPARACIÓN: ESCALA ORIGINAL Y LOGARÍTMICA =====\n")
print(
  bind_rows(resumen_original, resumen_log) %>%
    arrange(MAE_medio),
  digits = 4,
  row.names = FALSE
)

Fold 1 completado.
Fold 2 completado.
Fold 3 completado.

===== RESULTADOS LOGARÍTMICOS POR VENTANA =====
 fold                  modelo vuelos_validacion   MAE  RMSE
    1  Regresión lineal log1p               981 33.52 51.87
    1 Gradient Boosting log1p               981 32.20 49.31
    1     Random Forest log1p               981 33.73 50.64
    2  Regresión lineal log1p               991 36.90 55.69
    2 Gradient Boosting log1p               991 23.04 45.52
    2     Random Forest log1p               991 23.83 46.46
    3  Regresión lineal log1p               879 24.68 56.39
    3 Gradient Boosting log1p               879 24.25 56.23
    3     Random Forest log1p               879 24.61 56.35

===== COMPARACIÓN: ESCALA ORIGINAL Y LOGARÍTMICA =====
# A tibble: 7 × 6
  modelo                  folds MAE_medio MAE_sd RMSE_medio RMSE_sd
  <chr>                   <int>     <dbl>  <dbl>      <dbl>   <dbl>
1 Referencia: mediana         3      25.8   5.63       51.9    5.78
2 Gradient Boost

# Siguiente paso: extraer pesos e **importancia**

In [ ]:
# ============================================================
# AEROPREDICT — PESOS E IMPORTANCIA DE VARIABLES
# ============================================================

# 1. Importancia del clasificador GBM actual
cat("\n===== IMPORTANCIA: CLASIFICADOR GBM =====\n")

importancia_clasificacion <- summary(
  modelo_gbm_final,
  n.trees = 300,
  plotit = FALSE
)

print(head(importancia_clasificacion, 20), row.names = FALSE)


# 2. Importancia del GBM de regresión logarítmica.
# ATENCIÓN: corresponde al último fold de validación,
# no es todavía un modelo definitivo.
if (exists("gbm_log")) {
  cat("\n===== IMPORTANCIA: GBM REGRESIÓN LOGARÍTMICA, FOLD 3 =====\n")

  importancia_regresion_log <- summary(
    gbm_log,
    n.trees = 300,
    plotit = FALSE
  )

  print(head(importancia_regresion_log, 20), row.names = FALSE)
}


# 3. Coeficientes de la regresión lineal actual
cat("\n===== COEFICIENTES LINEALES: VARIABLES NUMÉRICAS =====\n")

coeficientes <- modelo_lm_regresion$coefficients

variables_numericas <- c(
  "temperatura_metar",
  "punto_rocio_metar",
  "relh"
)

print(
  coeficientes[
    intersect(variables_numericas, names(coeficientes))
  ]
)


# 4. Coeficientes categóricos de mayor magnitud.
# Son diferencias frente a categorías de referencia;
# no deben interpretarse directamente como importancia.
cat("\n===== COEFICIENTES CATEGÓRICOS DE MAYOR MAGNITUD =====\n")

coef_df <- data.frame(
  termino = names(coeficientes),
  coeficiente = as.numeric(coeficientes),
  row.names = NULL
)

coef_df <- coef_df[
  is.finite(coef_df$coeficiente) &
    coef_df$termino != "Intercept",
]

print(
  head(
    coef_df[
      order(abs(coef_df$coeficiente), decreasing = TRUE),
    ],
    15
  ),
  row.names = FALSE
)

cat("\n===== FIN =====\n")


===== IMPORTANCIA: CLASIFICADOR GBM =====
                   var    rel.inf
     punto_rocio_metar 25.4576129
     temperatura_metar 16.6356713
           dia_semana5 14.0619939
                  relh  8.4428215
           dia_semana4  6.4338181
           dia_semana6  6.1350521
           dia_semana2  4.1619404
           hora_vuelo7  3.7675683
           dia_semana3  3.4216880
           hora_vuelo6  1.9113703
           dia_semana7  1.3050442
           hora_vuelo8  1.0499353
          hora_vuelo15  0.9515221
        airline_iataFR  0.7684997
          hora_vuelo20  0.7207321
          hora_vuelo19  0.5810496
          hora_vuelo18  0.4501894
          hora_vuelo23  0.4397482
 other_airport_iataTLV  0.4337796
 other_airport_iataMIA  0.4093776

===== IMPORTANCIA: GBM REGRESIÓN LOGARÍTMICA, FOLD 3 =====
                   var    rel.inf
     temperatura_metar 17.2020832
     punto_rocio_metar 15.4757381
           dia_semana6 12.5948567
                  relh  8.8341696
        airli

# **3. Siguiente paso: comprobar la estabilidad con bootstrap**

In [ ]:
# ============================================================
# AEROPREDICT — PRUEBA LIGERA DE ESTABILIDAD
# 8 modelos, 100 árboles, submuestreo de días de TRAIN
# ============================================================

set.seed(2026)
B <- 8
N_TREES <- 100

predictores <- setdiff(names(datos_gbm_train), "retraso_15_min")

fechas_boot <- as.character(as.Date(train_clas$fecha_vuelo))
if (length(fechas_boot) != nrow(datos_gbm_train)) {
  stop("Las fechas y los datos de entrenamiento no están alineados.")
}
if (all(is.na(fechas_boot))) {
  stop("No hay fechas válidas para hacer el submuestreo temporal.")
}

dias <- unique(fechas_boot[!is.na(fechas_boot)])
resultados_estabilidad <- list()

for (b in seq_len(B)) {
  # Seleccionar la mitad de los días, sin reemplazo.
  set.seed(2026 + b)
  dias_sel <- sample(dias, max(2, ceiling(length(dias) / 2)))

  indices <- which(!is.na(fechas_boot) & fechas_boot %in% dias_sel)
  datos_b <- datos_gbm_train[indices, , drop = FALSE]

  # Quitar predictores constantes en esta muestra.
  vars_ok <- vapply(
    datos_b[, predictores, drop = FALSE],
    function(x) length(unique(x[!is.na(x)])) > 1,
    logical(1)
  )

  vars <- predictores[vars_ok]

  if (length(unique(datos_b$retraso_15_min)) < 2 || length(vars) == 0) {
    cat("Repetición", b, "omitida: muestra no válida.\n")
    next
  }

  datos_fit <- datos_b[, c("retraso_15_min", vars), drop = FALSE]

  ajuste <- tryCatch(
    gbm::gbm(
      retraso_15_min ~ .,
      data = datos_fit,
      distribution = "bernoulli",
      n.trees = N_TREES,
      interaction.depth = 3,
      shrinkage = 0.03,
      n.minobsinnode = 20,
      bag.fraction = 0.5,
      train.fraction = 1,
      verbose = FALSE
    ),
    error = function(e) {
      message("Repetición ", b, ": ", conditionMessage(e))
      NULL
    }
  )

  if (is.null(ajuste)) next

  imp <- summary(ajuste, n.trees = N_TREES, plotit = FALSE)
  v <- as.character(imp$var)

  grupo <- ifelse(v == "temperatura_metar", "temperatura_metar",
           ifelse(v == "punto_rocio_metar", "punto_rocio_metar",
           ifelse(v == "relh", "relh",
           ifelse(grepl("^dia_semana", v), "dia_semana",
           ifelse(grepl("^hora_vuelo", v), "hora_vuelo",
           ifelse(grepl("^airline_iata", v), "airline_iata",
           ifelse(grepl("^other_airport_iata", v),
                  "other_airport_iata", "otras")))))))

  agregado <- aggregate(
    rel.inf ~ grupo,
    data = data.frame(grupo = grupo, rel.inf = imp$rel.inf),
    FUN = sum
  )
  agregado$repeticion <- b
  resultados_estabilidad[[length(resultados_estabilidad) + 1]] <- agregado

  cat("Repetición", b, "de", B, "completada.\n")
  flush.console()
}

if (length(resultados_estabilidad) < 2) {
  stop("No hay suficientes repeticiones válidas para resumir la estabilidad.")
}

tabla_estabilidad <- do.call(rbind, resultados_estabilidad)

resumen_estabilidad <- do.call(rbind, lapply(
  unique(tabla_estabilidad$grupo),
  function(g) {
    z <- tabla_estabilidad$rel.inf[tabla_estabilidad$grupo == g]
    data.frame(
      variable = g,
      importancia_mediana = median(z),
      percentil_2_5 = unname(quantile(z, 0.025)),
      percentil_97_5 = unname(quantile(z, 0.975))
    )
  }
))

resumen_estabilidad <- resumen_estabilidad[
  order(-resumen_estabilidad$importancia_mediana), ]

cat("\n===== RESULTADO: ESTABILIDAD POR SUBMUESTREO =====\n")
cat("Repeticiones completadas:", length(resultados_estabilidad), "de", B, "\n")
print(resumen_estabilidad, row.names = FALSE, digits = 4)

Repetición 1 de 8 completada.
Repetición 2 de 8 completada.
Repetición 3 de 8 completada.
Repetición 4 de 8 completada.
Repetición 5 de 8 completada.
Repetición 6 de 8 completada.
Repetición 7 de 8 completada.
Repetición 8 de 8 completada.

===== RESULTADO: ESTABILIDAD POR SUBMUESTREO =====
Repeticiones completadas: 8 de 8 
           variable importancia_mediana percentil_2_5 percentil_97_5
         dia_semana            46.11935       21.9445        67.4684
  temperatura_metar            27.86976        6.1681        38.9789
         hora_vuelo            13.01327        5.9138        23.8663
  punto_rocio_metar             6.36141        1.4719        25.5889
               relh             5.55264        0.9603         9.2173
 other_airport_iata             0.46173        0.0000         0.6737
       airline_iata             0.09925        0.0000         0.6841


# **Siguiente paso: hacer lo mismo con la regresión**

In [ ]:
# ============================================================
# AEROPREDICT — ESTABILIDAD DE IMPORTANCIAS DE REGRESIÓN
# GBM sobre log1p(retraso_minutos)
# ============================================================

set.seed(3030)
B <- 8
N_TREES <- 100

# Recuperar las fechas de las filas de entrenamiento de regresión.
fechas_reg <- datos_train %>%
  dplyr::filter(retraso_15_min == 1) %>%
  dplyr::pull(fecha_vuelo)

fechas_reg <- as.character(as.Date(fechas_reg))

if (length(fechas_reg) != nrow(train_reg)) {
  stop("Las fechas no están alineadas con train_reg. No se continúa.")
}

dias <- unique(fechas_reg[!is.na(fechas_reg)])

# Preparar objetivo logarítmico y matriz de predictores existente.
datos_reg_gbm <- data.frame(
  retraso_log = log1p(train_reg$retraso_minutos),
  as.data.frame(x_train_reg),
  check.names = TRUE
)

predictores <- setdiff(names(datos_reg_gbm), "retraso_log")
resultados_reg <- list()

for (b in seq_len(B)) {

  set.seed(3030 + b)

  # Seleccionar la mitad de los días sin reemplazo.
  dias_sel <- sample(
    dias,
    size = max(2, ceiling(length(dias) / 2)),
    replace = FALSE
  )

  indices <- which(!is.na(fechas_reg) & fechas_reg %in% dias_sel)
  muestra <- datos_reg_gbm[indices, , drop = FALSE]

  # Eliminar columnas constantes en esta submuestra.
  variables_ok <- vapply(
    muestra[, predictores, drop = FALSE],
    function(x) length(unique(x[!is.na(x)])) > 1,
    logical(1)
  )

  vars <- predictores[variables_ok]

  if (length(indices) < 50 || length(vars) == 0) {
    cat("Repetición", b, "omitida por tamaño o variación insuficiente.\n")
    next
  }

  muestra_fit <- muestra[, c("retraso_log", vars), drop = FALSE]

  ajuste <- tryCatch(
    gbm::gbm(
      retraso_log ~ .,
      data = muestra_fit,
      distribution = "gaussian",
      n.trees = N_TREES,
      interaction.depth = 3,
      shrinkage = 0.03,
      n.minobsinnode = 20,
      bag.fraction = 0.5,
      train.fraction = 1,
      verbose = FALSE
    ),
    error = function(e) {
      message("Repetición ", b, ": ", conditionMessage(e))
      NULL
    }
  )

  if (is.null(ajuste)) next

  imp <- summary(ajuste, n.trees = N_TREES, plotit = FALSE)
  v <- as.character(imp$var)

  grupo <- ifelse(v == "temperatura_metar", "temperatura_metar",
           ifelse(v == "punto_rocio_metar", "punto_rocio_metar",
           ifelse(v == "relh", "relh",
           ifelse(grepl("^dia_semana", v), "dia_semana",
           ifelse(grepl("^hora_vuelo", v), "hora_vuelo",
           ifelse(grepl("^airline_iata", v), "airline_iata",
           ifelse(grepl("^other_airport_iata", v),
                  "other_airport_iata", "otras")))))))

  agregado <- aggregate(
    rel.inf ~ grupo,
    data = data.frame(grupo = grupo, rel.inf = imp$rel.inf),
    FUN = sum
  )

  agregado$repeticion <- b
  resultados_reg[[length(resultados_reg) + 1]] <- agregado

  cat("Regresión: repetición", b, "de", B, "completada.\n")
  flush.console()
}

if (length(resultados_reg) < 2) {
  stop("No hay suficientes repeticiones válidas.")
}

tabla_reg <- do.call(rbind, resultados_reg)

resumen_reg <- do.call(rbind, lapply(
  unique(tabla_reg$grupo),
  function(g) {
    z <- tabla_reg$rel.inf[tabla_reg$grupo == g]
    data.frame(
      variable = g,
      importancia_mediana = median(z),
      percentil_2_5 = unname(quantile(z, 0.025)),
      percentil_97_5 = unname(quantile(z, 0.975))
    )
  }
))

resumen_reg <- resumen_reg[
  order(-resumen_reg$importancia_mediana), ]

cat("\n===== ESTABILIDAD DE IMPORTANCIAS: REGRESIÓN =====\n")
cat("Repeticiones completadas:", length(resultados_reg), "de", B, "\n")
print(resumen_reg, row.names = FALSE, digits = 4)

Regresión: repetición 1 de 8 completada.
Regresión: repetición 2 de 8 completada.
Regresión: repetición 3 de 8 completada.
Regresión: repetición 4 de 8 completada.
Regresión: repetición 5 de 8 completada.
Regresión: repetición 6 de 8 completada.
Regresión: repetición 7 de 8 completada.
Regresión: repetición 8 de 8 completada.

===== ESTABILIDAD DE IMPORTANCIAS: REGRESIÓN =====
Repeticiones completadas: 8 de 8 
           variable importancia_mediana percentil_2_5 percentil_97_5
         dia_semana               28.19         1.928          43.56
  temperatura_metar               15.89         8.391          28.83
         hora_vuelo               13.16         7.642          20.27
  punto_rocio_metar               12.87         6.108          30.56
               relh               10.89         5.384          34.96
 other_airport_iata               10.86         6.698          18.01
       airline_iata                3.98         2.702           5.55


# **1. Validación temporal del clasificador**

In [ ]:
# ============================================================
# AEROPREDICT — VALIDACIÓN TEMPORAL DEL CLASIFICADOR
# Solo TRAIN. No utiliza el TEST final.
# ============================================================

set.seed(2026)

UMBRAL_CANDIDATO <- 0.08
N_TREES <- 300

# Comprobar objetos necesarios
stopifnot(
  exists("datos_train"),
  exists("variables_predictoras"),
  exists("variables_categoricas"),
  exists("preparar_categorias_temporal"),
  exists("crear_matriz_predictores")
)

base_cv <- datos_train
base_cv$fecha_vuelo <- as.Date(base_cv$fecha_vuelo)

if (anyNA(base_cv$fecha_vuelo)) {
  stop("Hay fechas ausentes en TRAIN. Hay que revisarlas antes de continuar.")
}

fechas <- sort(unique(base_cv$fecha_vuelo))
n_fechas <- length(fechas)

if (n_fechas < 10) {
  stop("No hay suficientes fechas para realizar la validación temporal.")
}

# Tres ventanas temporales expansivas:
# 1) entrena hasta el 40%, valida del 40% al 60%
# 2) entrena hasta el 60%, valida del 60% al 80%
# 3) entrena hasta el 80%, valida el 20% restante
plan <- data.frame(
  fold = 1:3,
  proporcion_train = c(0.40, 0.60, 0.80),
  proporcion_fin_validacion = c(0.60, 0.80, 1.00)
)

# Métricas de clasificación y ranking
calcular_metricas_cv <- function(real, prob, umbral) {
  real <- as.integer(as.character(real))
  valido <- is.finite(prob) & !is.na(real)
  real <- real[valido]
  prob <- prob[valido]

  pred <- as.integer(prob >= umbral)

  tp <- sum(pred == 1 & real == 1)
  tn <- sum(pred == 0 & real == 0)
  fp <- sum(pred == 1 & real == 0)
  fn <- sum(pred == 0 & real == 1)

  precision <- if ((tp + fp) > 0) tp / (tp + fp) else 0
  recall <- if ((tp + fn) > 0) tp / (tp + fn) else 0
  especificidad <- if ((tn + fp) > 0) tn / (tn + fp) else 0
  f1 <- if ((precision + recall) > 0) {
    2 * precision * recall / (precision + recall)
  } else {
    0
  }

  n_pos <- sum(real == 1)
  n_neg <- sum(real == 0)

  auc <- NA_real_
  ap <- NA_real_

  if (n_pos > 0 && n_neg > 0) {
    rangos <- rank(prob, ties.method = "average")
    auc <- (sum(rangos[real == 1]) - n_pos * (n_pos + 1) / 2) /
      (n_pos * n_neg)

    orden <- order(prob, decreasing = TRUE)
    y_ordenado <- real[orden]
    precision_acumulada <- cumsum(y_ordenado) / seq_along(y_ordenado)
    ap <- mean(precision_acumulada[y_ordenado == 1])
  }

  data.frame(
    AUC = auc,
    AP = ap,
    precision = precision,
    recall = recall,
    especificidad = especificidad,
    F1 = f1,
    pct_predichos_positivos = mean(pred == 1) * 100,
    prevalencia_real = mean(real == 1) * 100
  )
}

resultados_cv_clasificacion <- list()

for (i in seq_len(nrow(plan))) {

  fin_train <- max(1, floor(n_fechas * plan$proporcion_train[i]))
  fin_valid <- min(
    n_fechas,
    max(fin_train + 1,
        floor(n_fechas * plan$proporcion_fin_validacion[i]))
  )

  if (fin_train >= fin_valid) {
    cat("Ventana", i, "omitida por falta de fechas.\n")
    next
  }

  fechas_train_fold <- fechas[seq_len(fin_train)]
  fechas_valid_fold <- fechas[(fin_train + 1):fin_valid]

  train_fold <- base_cv[
    base_cv$fecha_vuelo %in% fechas_train_fold, , drop = FALSE
  ]
  valid_fold <- base_cv[
    base_cv$fecha_vuelo %in% fechas_valid_fold, , drop = FALSE
  ]

  if (nrow(train_fold) == 0 || nrow(valid_fold) == 0) next

  y_train <- as.integer(as.character(train_fold$retraso_15_min))
  y_valid <- as.integer(as.character(valid_fold$retraso_15_min))

  if (length(unique(y_train)) < 2 ||
      length(unique(y_valid)) < 2) {
    cat("Ventana", i, "omitida: falta una de las clases.\n")
    next
  }

  # El preprocesamiento aprende las categorías solo desde TRAIN.
  prep <- preparar_categorias_temporal(
    train_fold,
    valid_fold,
    valid_fold,
    variables_categoricas = variables_categoricas,
    frecuencia_minima = 2
  )

  x_train_completo <- crear_matriz_predictores(
    prep$train, formula_x
  )

  # Retirar columnas constantes en la ventana de entrenamiento.
  variables_ok <- vapply(
    seq_len(ncol(x_train_completo)),
    function(j) length(unique(x_train_completo[, j])) > 1,
    logical(1)
  )

  columnas <- colnames(x_train_completo)[variables_ok]

  if (length(columnas) == 0) {
    cat("Ventana", i, "omitida: no hay predictores variables.\n")
    next
  }

  x_train <- x_train_completo[, columnas, drop = FALSE]
  x_valid <- crear_matriz_predictores(
    prep$validacion, formula_x, columnas
  )

  datos_fit <- data.frame(
    retraso_15_min = y_train,
    as.data.frame(x_train),
    check.names = TRUE
  )

  set.seed(9000 + i)

  modelo_cv <- gbm::gbm(
    retraso_15_min ~ .,
    data = datos_fit,
    distribution = "bernoulli",
    n.trees = N_TREES,
    interaction.depth = 3,
    shrinkage = 0.03,
    n.minobsinnode = 20,
    bag.fraction = 0.5,
    train.fraction = 1,
    verbose = FALSE
  )

  prob <- as.numeric(predict(
    modelo_cv,
    newdata = as.data.frame(x_valid),
    n.trees = N_TREES,
    type = "response"
  ))

  met_gbm <- calcular_metricas_cv(
    y_valid, prob, UMBRAL_CANDIDATO
  )
  met_gbm$ventana <- i
  met_gbm$modelo <- "GBM"

  # Referencia: clasificar todos los vuelos como retrasados.
  p <- mean(y_valid == 1)
  met_ref <- data.frame(
    AUC = 0.5,
    AP = p,
    precision = p,
    recall = 1,
    especificidad = 0,
    F1 = 2 * p / (1 + p),
    pct_predichos_positivos = 100,
    prevalencia_real = p * 100,
    ventana = i,
    modelo = "Referencia: todos positivos"
  )

  resultados_cv_clasificacion[[length(resultados_cv_clasificacion) + 1]] <-
    rbind(met_gbm, met_ref)

  cat(
    "Ventana", i, "completada | TRAIN:", nrow(train_fold),
    "| VALIDACIÓN:", nrow(valid_fold), "\n"
  )
  flush.console()
}

if (length(resultados_cv_clasificacion) < 2) {
  stop("No se completaron suficientes ventanas. Revisa los resultados anteriores.")
}

tabla_cv_clasificacion <- do.call(
  rbind, resultados_cv_clasificacion
)

cat("\n===== RESULTADOS POR VENTANA =====\n")
print(
  tabla_cv_clasificacion[
    , c("ventana", "modelo", "AUC", "AP", "precision",
        "recall", "especificidad", "F1",
        "pct_predichos_positivos", "prevalencia_real")
  ],
  row.names = FALSE,
  digits = 4
)

metricas <- c(
  "AUC", "AP", "precision", "recall",
  "especificidad", "F1", "pct_predichos_positivos"
)

resumen_cv <- do.call(rbind, lapply(
  unique(tabla_cv_clasificacion$modelo),
  function(m) {
    d <- tabla_cv_clasificacion[
      tabla_cv_clasificacion$modelo == m, , drop = FALSE
    ]

    fila <- data.frame(
      modelo = m,
      ventanas = nrow(d)
    )

    for (met in metricas) {
      fila[[paste0(met, "_media")]] <- mean(d[[met]], na.rm = TRUE)
      fila[[paste0(met, "_sd")]] <- sd(d[[met]], na.rm = TRUE)
    }

    fila
  }
))

cat("\n===== RESUMEN TEMPORAL =====\n")
print(resumen_cv, row.names = FALSE, digits = 4)

Ventana 1 completada | TRAIN: 18120 | VALIDACIÓN: 10137 
Ventana 2 completada | TRAIN: 28257 | VALIDACIÓN: 6206 
Ventana 3 completada | TRAIN: 34463 | VALIDACIÓN: 10032 

===== RESULTADOS POR VENTANA =====
 ventana                      modelo    AUC      AP precision recall
       1                         GBM 0.3179 0.02956   0.04484 0.8218
       1 Referencia: todos positivos 0.5000 0.04429   0.04429 1.0000
       2                         GBM 0.2506 0.09903   0.05828 0.2074
       2 Referencia: todos positivos 0.5000 0.15694   0.15694 1.0000
       3                         GBM 0.4919 0.11809   0.07525 0.4545
       3 Referencia: todos positivos 0.5000 0.09759   0.09759 1.0000
 especificidad      F1 pct_predichos_positivos prevalencia_real
        0.1886 0.08503                   81.19            4.429
        0.0000 0.08483                  100.00            4.429
        0.3761 0.09099                   55.85           15.694
        0.0000 0.27131                  100.00         

In [ ]:
# ============================================================
# DIAGNÓSTICO DEL CLASIFICADOR GBM
# ¿Problema de entrenamiento, inversión de puntuaciones
# o cambio temporal entre entrenamiento y validación?
# ============================================================

auc_diagnostico <- function(real, puntuacion) {
  ok <- !is.na(real) & is.finite(puntuacion)
  real <- as.integer(real[ok])
  puntuacion <- puntuacion[ok]

  n1 <- sum(real == 1)
  n0 <- sum(real == 0)

  if (n1 == 0 || n0 == 0) return(NA_real_)

  r <- rank(puntuacion, ties.method = "average")
  (sum(r[real == 1]) - n1 * (n1 + 1) / 2) / (n1 * n0)
}

resultados_diagnostico <- list()

for (i in seq_len(nrow(plan))) {

  fin_train <- floor(n_fechas * plan$proporcion_train[i])
  fin_valid <- min(
    n_fechas,
    floor(n_fechas * plan$proporcion_fin_validacion[i])
  )

  if (fin_train < 1 || fin_valid <= fin_train) next

  fechas_train_fold <- fechas[seq_len(fin_train)]
  fechas_valid_fold <- fechas[(fin_train + 1):fin_valid]

  train_fold <- base_cv[
    base_cv$fecha_vuelo %in% fechas_train_fold, , drop = FALSE
  ]
  valid_fold <- base_cv[
    base_cv$fecha_vuelo %in% fechas_valid_fold, , drop = FALSE
  ]

  y_train <- as.integer(as.character(train_fold$retraso_15_min))
  y_valid <- as.integer(as.character(valid_fold$retraso_15_min))

  if (length(unique(y_train)) < 2 ||
      length(unique(y_valid)) < 2) next

  prep <- preparar_categorias_temporal(
    train_fold,
    valid_fold,
    valid_fold,
    variables_categoricas = variables_categoricas,
    frecuencia_minima = 2
  )

  x_train_completo <- crear_matriz_predictores(prep$train, formula_x)

  var_ok <- vapply(
    seq_len(ncol(x_train_completo)),
    function(j) length(unique(x_train_completo[, j])) > 1,
    logical(1)
  )

  columnas <- colnames(x_train_completo)[var_ok]
  if (length(columnas) == 0) next

  x_train <- x_train_completo[, columnas, drop = FALSE]
  x_valid <- crear_matriz_predictores(
    prep$validacion, formula_x, columnas
  )

  datos_fit <- data.frame(
    retraso_15_min = y_train,
    as.data.frame(x_train),
    check.names = TRUE
  )

  set.seed(9000 + i)

  modelo_diag <- gbm::gbm(
    retraso_15_min ~ .,
    data = datos_fit,
    distribution = "bernoulli",
    n.trees = 300,
    interaction.depth = 3,
    shrinkage = 0.03,
    n.minobsinnode = 20,
    bag.fraction = 0.5,
    train.fraction = 1,
    verbose = FALSE
  )

  puntuacion_train <- as.numeric(predict(
    modelo_diag,
    newdata = as.data.frame(x_train),
    n.trees = 300,
    type = "response"
  ))

  puntuacion_valid <- as.numeric(predict(
    modelo_diag,
    newdata = as.data.frame(x_valid),
    n.trees = 300,
    type = "response"
  ))

  resultados_diagnostico[[length(resultados_diagnostico) + 1]] <-
    data.frame(
      ventana = i,
      filas_train = length(y_train),
      filas_validacion = length(y_valid),
      prevalencia_train_pct = mean(y_train == 1) * 100,
      prevalencia_validacion_pct = mean(y_valid == 1) * 100,
      auc_train = auc_diagnostico(y_train, puntuacion_train),
      auc_validacion = auc_diagnostico(y_valid, puntuacion_valid),
      auc_validacion_invertida = auc_diagnostico(
        y_valid, 1 - puntuacion_valid
      ),
      media_score_validacion_retraso = mean(
        puntuacion_valid[y_valid == 1]
      ),
      media_score_validacion_sin_retraso = mean(
        puntuacion_valid[y_valid == 0]
      )
    )

  cat("Diagnóstico de ventana", i, "completado.\n")
}

tabla_diagnostico <- do.call(rbind, resultados_diagnostico)

cat("\n===== DIAGNÓSTICO DE AUC Y PUNTUACIONES =====\n")
print(tabla_diagnostico, row.names = FALSE, digits = 4)

Diagnóstico de ventana 1 completado.
Diagnóstico de ventana 2 completado.
Diagnóstico de ventana 3 completado.

===== DIAGNÓSTICO DE AUC Y PUNTUACIONES =====
 ventana filas_train filas_validacion prevalencia_train_pct
       1       18120            10137                 20.91
       2       28257             6206                 15.00
       3       34463            10032                 15.12
 prevalencia_validacion_pct auc_train auc_validacion auc_validacion_invertida
                      4.429    0.8716         0.3179                   0.6821
                     15.694    0.8802         0.2506                   0.7494
                      9.759    0.8573         0.4919                   0.5081
 media_score_validacion_retraso media_score_validacion_sin_retraso
                        0.20000                            0.34382
                        0.06196                            0.14768
                        0.10152                            0.09996


# **3. Vamos a comprobar primero las etiquetas**

In [ ]:
# ============================================================
# AEROPREDICT — AUDITORÍA DE ETIQUETAS Y DISTRIBUCIÓN TEMPORAL
# No entrena modelos ni modifica el TEST.
# ============================================================

library(dplyr)
library(lubridate)

stopifnot(
  "retraso_15_min" %in% names(df_modelado),
  "retraso_minutos" %in% names(df_modelado),
  "fecha_vuelo" %in% names(df_modelado)
)

auditoria <- df_modelado %>%
  mutate(
    objetivo_num = suppressWarnings(
      as.integer(as.character(retraso_15_min))
    ),
    objetivo_esperado = ifelse(
      is.na(retraso_minutos),
      NA_integer_,
      as.integer(retraso_minutos >= 15)
    ),
    fecha_vuelo = as.Date(fecha_vuelo)
  )

cat("\n===== 1. TIPO Y VALORES DE LA ETIQUETA =====\n")
print(class(df_modelado$retraso_15_min))
print(table(df_modelado$retraso_15_min, useNA = "ifany"))

cat("\n===== 2. COHERENCIA DE LA ETIQUETA =====\n")
print(
  auditoria %>%
    summarise(
      filas = n(),
      etiquetas_ausentes = sum(is.na(objetivo_num)),
      minutos_ausentes = sum(is.na(retraso_minutos)),
      etiquetas_incoherentes = sum(
        !is.na(objetivo_num) &
        !is.na(objetivo_esperado) &
        objetivo_num != objetivo_esperado
      ),
      prevalencia_pct = mean(objetivo_num == 1, na.rm = TRUE) * 100
    )
)

cat("\n===== 3. PREVALENCIA POR SEMANA =====\n")
prevalencia_semanal <- auditoria %>%
  filter(!is.na(fecha_vuelo), !is.na(objetivo_num)) %>%
  mutate(semana = floor_date(fecha_vuelo, unit = "week")) %>%
  group_by(semana) %>%
  summarise(
    vuelos = n(),
    retrasos_pct = mean(objetivo_num == 1) * 100,
    .groups = "drop"
  ) %>%
  arrange(semana)

print(prevalencia_semanal, n = Inf)

cat("\n===== 4. FECHAS DEL CONJUNTO MODELADO =====\n")
print(range(auditoria$fecha_vuelo, na.rm = TRUE))


Attaching package: ‘lubridate’


The following objects are masked from ‘package:base’:

    date, intersect, setdiff, union





===== 1. TIPO Y VALORES DE LA ETIQUETA =====
[1] "integer"

    0     1 
70341 13066 

===== 2. COHERENCIA DE LA ETIQUETA =====
  filas etiquetas_ausentes minutos_ausentes etiquetas_incoherentes
1 83407                  0                0                      0
  prevalencia_pct
1        15.66535

===== 3. PREVALENCIA POR SEMANA =====
# A tibble: 9 × 3
  semana     vuelos retrasos_pct
  <date>      <int>        <dbl>
1 2026-07-26    860        68.4 
2 2026-08-02  20745        15.4 
3 2026-08-23   6652         6.75
4 2026-08-30   2555         0   
5 2026-09-13   6977        14.0 
6 2026-09-20  13588         7.29
7 2026-10-04    100        46   
8 2026-10-11  24413        16.9 
9 2026-10-18   7517        35.9 

===== 4. FECHAS DEL CONJUNTO MODELADO =====
[1] "2026-08-01" "2026-10-19"


# **Siguiente prueba: GBM frente a regresión logística**

In [ ]:
# ============================================================
# AEROPREDICT — COMPARACIÓN GBM vs REGRESIÓN LOGÍSTICA
# Misma validación temporal, sin utilizar TEST FINAL
# ============================================================

auc_modelo <- function(real, score) {
  ok <- !is.na(real) & is.finite(score)
  real <- as.integer(real[ok])
  score <- score[ok]

  n1 <- sum(real == 1)
  n0 <- sum(real == 0)
  if (n1 == 0 || n0 == 0) return(NA_real_)

  r <- rank(score, ties.method = "average")
  (sum(r[real == 1]) - n1 * (n1 + 1) / 2) / (n1 * n0)
}

ap_modelo <- function(real, score) {
  ok <- !is.na(real) & is.finite(score)
  real <- as.integer(real[ok])
  score <- score[ok]

  if (sum(real == 1) == 0) return(NA_real_)

  orden <- order(score, decreasing = TRUE)
  y <- real[orden]
  precision_acum <- cumsum(y) / seq_along(y)

  mean(precision_acum[y == 1])
}

resultados_comparacion <- list()

for (i in seq_len(nrow(plan))) {

  fin_train <- floor(n_fechas * plan$proporcion_train[i])
  fin_valid <- min(
    n_fechas,
    floor(n_fechas * plan$proporcion_fin_validacion[i])
  )

  if (fin_train < 1 || fin_valid <= fin_train) next

  fechas_train_fold <- fechas[seq_len(fin_train)]
  fechas_valid_fold <- fechas[(fin_train + 1):fin_valid]

  train_fold <- base_cv[
    base_cv$fecha_vuelo %in% fechas_train_fold, , drop = FALSE
  ]
  valid_fold <- base_cv[
    base_cv$fecha_vuelo %in% fechas_valid_fold, , drop = FALSE
  ]

  y_train <- as.integer(as.character(train_fold$retraso_15_min))
  y_valid <- as.integer(as.character(valid_fold$retraso_15_min))

  if (length(unique(y_train)) < 2 ||
      length(unique(y_valid)) < 2) next

  prep <- preparar_categorias_temporal(
    train_fold,
    valid_fold,
    valid_fold,
    variables_categoricas = variables_categoricas,
    frecuencia_minima = 2
  )

  x_completo <- crear_matriz_predictores(prep$train, formula_x)

  variables_ok <- vapply(
    seq_len(ncol(x_completo)),
    function(j) length(unique(x_completo[, j])) > 1,
    logical(1)
  )

  columnas <- colnames(x_completo)[variables_ok]
  if (length(columnas) == 0) next

  x_train <- x_completo[, columnas, drop = FALSE]
  x_valid <- crear_matriz_predictores(
    prep$validacion, formula_x, columnas
  )

  # GBM
  datos_fit <- data.frame(
    retraso_15_min = y_train,
    as.data.frame(x_train),
    check.names = TRUE
  )

  set.seed(9000 + i)
  modelo_gbm_cv <- gbm::gbm(
    retraso_15_min ~ .,
    data = datos_fit,
    distribution = "bernoulli",
    n.trees = 300,
    interaction.depth = 3,
    shrinkage = 0.03,
    n.minobsinnode = 20,
    bag.fraction = 0.5,
    train.fraction = 1,
    verbose = FALSE
  )

  score_gbm <- as.numeric(predict(
    modelo_gbm_cv,
    newdata = as.data.frame(x_valid),
    n.trees = 300,
    type = "response"
  ))

  # Regresión logística: matriz de diseño y selección de columnas
  # linealmente independientes para evitar redundancias exactas.
  X_train_completo <- cbind(
    Intercept = 1,
    as.matrix(x_train)
  )
  X_valid_completo <- cbind(
    Intercept = 1,
    as.matrix(x_valid)
  )

  qr_x <- qr(X_train_completo)
  indices_independientes <- qr_x$pivot[seq_len(qr_x$rank)]
  columnas_glm <- colnames(X_train_completo)[indices_independientes]

  X_train_glm <- X_train_completo[
    , columnas_glm, drop = FALSE
  ]
  X_valid_glm <- X_valid_completo[
    , columnas_glm, drop = FALSE
  ]

  modelo_glm <- tryCatch(
    suppressWarnings(
      glm.fit(
        x = X_train_glm,
        y = y_train,
        family = binomial(),
        control = glm.control(maxit = 50)
      )
    ),
    error = function(e) {
      message("La logística falló en ventana ", i, ": ",
              conditionMessage(e))
      NULL
    }
  )

  if (is.null(modelo_glm)) next

  score_glm <- as.numeric(
    plogis(X_valid_glm %*% modelo_glm$coefficients)
  )

  if (any(!is.finite(score_glm))) {
    cat("Logística omitida en ventana", i,
        ": puntuaciones no finitas.\n")
    next
  }

  prevalencia <- mean(y_valid == 1)

  resultados_comparacion[[length(resultados_comparacion) + 1]] <-
    data.frame(
      ventana = i,
      modelo = c(
        "GBM",
        "GBM invertido (diagnóstico)",
        "Regresión logística",
        "Referencia: todos positivos"
      ),
      AUC = c(
        auc_modelo(y_valid, score_gbm),
        auc_modelo(y_valid, 1 - score_gbm),
        auc_modelo(y_valid, score_glm),
        0.5
      ),
      AP = c(
        ap_modelo(y_valid, score_gbm),
        ap_modelo(y_valid, 1 - score_gbm),
        ap_modelo(y_valid, score_glm),
        prevalencia
      ),
      prevalencia_pct = prevalencia * 100
    )

  cat("Comparación de ventana", i, "completada.\n")
  flush.console()
}

if (length(resultados_comparacion) == 0) {
  stop("No se completó ninguna comparación.")
}

tabla_comparacion <- do.call(rbind, resultados_comparacion)

cat("\n===== COMPARACIÓN POR VENTANA =====\n")
print(tabla_comparacion, row.names = FALSE, digits = 4)

cat("\n===== MEDIA DE MÉTRICAS POR MODELO =====\n")
resumen_comparacion <- aggregate(
  cbind(AUC, AP) ~ modelo,
  data = tabla_comparacion,
  FUN = function(x) mean(x, na.rm = TRUE)
)
print(resumen_comparacion, row.names = FALSE, digits = 4)

Comparación de ventana 1 completada.
Comparación de ventana 2 completada.
Comparación de ventana 3 completada.

===== COMPARACIÓN POR VENTANA =====
 ventana                      modelo    AUC      AP prevalencia_pct
       1                         GBM 0.3179 0.02956           4.429
       1 GBM invertido (diagnóstico) 0.6821 0.06471           4.429
       1         Regresión logística 0.5390 0.04948           4.429
       1 Referencia: todos positivos 0.5000 0.04429           4.429
       2                         GBM 0.2506 0.09903          15.694
       2 GBM invertido (diagnóstico) 0.7494 0.32173          15.694
       2         Regresión logística 0.3237 0.10831          15.694
       2 Referencia: todos positivos 0.5000 0.15694          15.694
       3                         GBM 0.4919 0.11809           9.759
       3 GBM invertido (diagnóstico) 0.5081 0.10211           9.759
       3         Regresión logística 0.5761 0.15464           9.759
       3 Referencia: todos positivos

In [ ]:
# ============================================================
# AEROPREDICT — LOGÍSTICA REDUCIDA CON VALIDACIÓN TEMPORAL
# Predictores: hora, día de la semana y meteorología
# ============================================================

variables_reducidas <- c(
  "hora_vuelo",
  "dia_semana",
  "temperatura_metar",
  "punto_rocio_metar",
  "relh"
)

categoricas_reducidas <- intersect(
  c("hora_vuelo", "dia_semana"),
  variables_reducidas
)

formula_reducida <- stats::as.formula(
  paste("~", paste(variables_reducidas, collapse = " + "))
)

resultados_log_reducida <- list()

for (i in seq_len(nrow(plan))) {

  fin_train <- floor(n_fechas * plan$proporcion_train[i])
  fin_valid <- min(
    n_fechas,
    floor(n_fechas * plan$proporcion_fin_validacion[i])
  )

  if (fin_train < 1 || fin_valid <= fin_train) next

  fechas_train_fold <- fechas[seq_len(fin_train)]
  fechas_valid_fold <- fechas[(fin_train + 1):fin_valid]

  train_fold <- base_cv[
    base_cv$fecha_vuelo %in% fechas_train_fold, , drop = FALSE
  ]
  valid_fold <- base_cv[
    base_cv$fecha_vuelo %in% fechas_valid_fold, , drop = FALSE
  ]

  y_train <- as.integer(as.character(train_fold$retraso_15_min))
  y_valid <- as.integer(as.character(valid_fold$retraso_15_min))

  if (length(unique(y_train)) < 2 ||
      length(unique(y_valid)) < 2) next

  prep <- preparar_categorias_temporal(
    train_fold,
    valid_fold,
    valid_fold,
    variables_categoricas = categoricas_reducidas,
    variables_agrupadas = character(0),
    frecuencia_minima = 2
  )

  # Seleccionar solamente las cinco variables del experimento.
  x_train_completo <- crear_matriz_predictores(
    prep$train[, variables_reducidas, drop = FALSE],
    formula_reducida
  )

  # Retirar columnas constantes y redundancias lineales exactas.
  constantes <- vapply(
    seq_len(ncol(x_train_completo)),
    function(j) length(unique(x_train_completo[, j])) <= 1,
    logical(1)
  )

  x_train_completo <- x_train_completo[
    , !constantes, drop = FALSE
  ]

  X_train_completo <- cbind(
    Intercept = 1,
    as.matrix(x_train_completo)
  )

  qr_x <- qr(X_train_completo)
  indices <- qr_x$pivot[seq_len(qr_x$rank)]
  columnas <- colnames(X_train_completo)[indices]
  X_train <- X_train_completo[, columnas, drop = FALSE]

  x_valid_completo <- crear_matriz_predictores(
    prep$validacion[, variables_reducidas, drop = FALSE],
    formula_reducida,
    colnames(x_train_completo)
  )

  X_valid_completo <- cbind(
    Intercept = 1,
    as.matrix(x_valid_completo)
  )
  X_valid <- X_valid_completo[, columnas, drop = FALSE]

  modelo <- tryCatch(
    suppressWarnings(
      glm.fit(
        x = X_train,
        y = y_train,
        family = binomial(),
        control = glm.control(maxit = 50)
      )
    ),
    error = function(e) {
      message("Ventana ", i, ": ", conditionMessage(e))
      NULL
    }
  )

  if (is.null(modelo) ||
      any(!is.finite(modelo$coefficients))) {
    cat("Ventana", i, "omitida: coeficientes no válidos.\n")
    next
  }

  score <- as.numeric(plogis(X_valid %*% modelo$coefficients))

  resultado <- data.frame(
    ventana = i,
    AUC = auc_modelo(y_valid, score),
    AP = ap_modelo(y_valid, score),
    prevalencia_pct = mean(y_valid == 1) * 100
  )

  resultados_log_reducida[[length(resultados_log_reducida) + 1]] <-
    resultado

  cat("Logística reducida: ventana", i, "completada.\n")
}

if (length(resultados_log_reducida) == 0) {
  stop("No se completó ninguna ventana.")
}

tabla_log_reducida <- do.call(rbind, resultados_log_reducida)

cat("\n===== LOGÍSTICA REDUCIDA POR VENTANA =====\n")
print(tabla_log_reducida, row.names = FALSE, digits = 4)

cat("\n===== MEDIA DE LAS VENTANAS =====\n")
print(
  data.frame(
    ventanas = nrow(tabla_log_reducida),
    AUC_media = mean(tabla_log_reducida$AUC, na.rm = TRUE),
    AP_media = mean(tabla_log_reducida$AP, na.rm = TRUE),
    AP_prevalencia_media = mean(
      tabla_log_reducida$prevalencia_pct, na.rm = TRUE
    )
  ),
  row.names = FALSE,
  digits = 4
)

Logística reducida: ventana 1 completada.
Logística reducida: ventana 2 completada.
Logística reducida: ventana 3 completada.

===== LOGÍSTICA REDUCIDA POR VENTANA =====
 ventana    AUC      AP prevalencia_pct
       1 0.5332 0.04458           4.429
       2 0.2708 0.10365          15.694
       3 0.5718 0.15998           9.759

===== MEDIA DE LAS VENTANAS =====
 ventanas AUC_media AP_media AP_prevalencia_media
        3    0.4586   0.1027                9.961


# **Paso 1. Entrenar, evaluar y guardar los modelos definitivos**

In [ ]:
# ============================================================
# AEROPREDICT — CIERRE DE MODELOS PARA LA ENTREGA
# Clasificación: regresión logística
# Regresión: GBM con objetivo log1p
# Entrenamiento final: TRAIN + VALIDACIÓN
# TEST FINAL: reservado para evaluación
# ============================================================

stopifnot(
  exists("datos_train"),
  exists("datos_val"),
  exists("datos_test_final"),
  exists("variables_predictoras"),
  exists("variables_categoricas"),
  exists("formula_x"),
  exists("preparar_categorias_temporal"),
  exists("crear_matriz_predictores")
)

set.seed(2026)

N_TREES_REG <- 300
UMBRAL_CLASIFICACION <- 0.5

# ------------------------------------------------------------
# 1. Unir TRAIN y VALIDACIÓN.
# TEST FINAL no participa en el entrenamiento.
# ------------------------------------------------------------

datos_desarrollo <- dplyr::bind_rows(datos_train, datos_val) %>%
  dplyr::arrange(fecha_vuelo)

cat("Filas para entrenamiento final:", nrow(datos_desarrollo), "\n")
cat("Filas reservadas para TEST:", nrow(datos_test_final), "\n")

# Métricas comunes
calcular_auc_final <- function(real, score) {
  ok <- !is.na(real) & is.finite(score)
  real <- as.integer(real[ok])
  score <- score[ok]

  positivos <- sum(real == 1)
  negativos <- sum(real == 0)

  if (positivos == 0 || negativos == 0) return(NA_real_)

  r <- rank(score, ties.method = "average")
  (sum(r[real == 1]) - positivos * (positivos + 1) / 2) /
    (positivos * negativos)
}

calcular_ap_final <- function(real, score) {
  ok <- !is.na(real) & is.finite(score)
  real <- as.integer(real[ok])
  score <- score[ok]

  if (sum(real == 1) == 0) return(NA_real_)

  orden <- order(score, decreasing = TRUE)
  y <- real[orden]
  precision_acumulada <- cumsum(y) / seq_along(y)

  mean(precision_acumulada[y == 1])
}

columnas_variables <- function(X) {
  conservar <- vapply(
    seq_len(ncol(X)),
    function(j) length(unique(X[, j])) > 1,
    logical(1)
  )

  colnames(X)[conservar]
}

# ------------------------------------------------------------
# 2. CLASIFICACIÓN: regresión logística
# ------------------------------------------------------------

cat("\n===== ENTRENAMIENTO DE CLASIFICACIÓN =====\n")

datos_clas_dev <- datos_desarrollo %>%
  dplyr::select(retraso_15_min, dplyr::all_of(variables_predictoras))

datos_clas_test <- datos_test_final %>%
  dplyr::select(retraso_15_min, dplyr::all_of(variables_predictoras))

prep_clas_final <- preparar_categorias_temporal(
  datos_clas_dev,
  datos_clas_dev,
  datos_clas_test,
  variables_categoricas = variables_categoricas,
  frecuencia_minima = 2
)

x_clas_dev_completo <- crear_matriz_predictores(
  prep_clas_final$train,
  formula_x
)

columnas_x_clas_final <- columnas_variables(x_clas_dev_completo)

if (length(columnas_x_clas_final) == 0) {
  stop("No hay variables válidas para entrenar la clasificación.")
}

x_clas_dev <- x_clas_dev_completo[
  , columnas_x_clas_final, drop = FALSE
]

x_clas_test <- crear_matriz_predictores(
  prep_clas_final$test,
  formula_x,
  columnas_x_clas_final
)

y_clas_dev <- as.integer(
  as.character(prep_clas_final$train$retraso_15_min)
)

y_clas_test <- as.integer(
  as.character(prep_clas_final$test$retraso_15_min)
)

if (length(unique(y_clas_dev)) != 2) {
  stop("La etiqueta de clasificación no contiene ambas clases.")
}

X_clas_dev_completo <- cbind(
  Intercept = 1,
  as.matrix(x_clas_dev)
)

X_clas_test_completo <- cbind(
  Intercept = 1,
  as.matrix(x_clas_test)
)

# Retirar redundancias lineales exactas
qr_clas <- qr(X_clas_dev_completo)
idx_clas <- qr_clas$pivot[seq_len(qr_clas$rank)]
columnas_glm_final <- colnames(X_clas_dev_completo)[idx_clas]

X_clas_dev <- X_clas_dev_completo[
  , columnas_glm_final, drop = FALSE
]

X_clas_test <- X_clas_test_completo[
  , columnas_glm_final, drop = FALSE
]

modelo_logistica_final <- suppressWarnings(
  stats::glm.fit(
    x = X_clas_dev,
    y = y_clas_dev,
    family = stats::binomial(),
    control = stats::glm.control(maxit = 100)
  )
)

if (any(!is.finite(modelo_logistica_final$coefficients))) {
  stop("La logística produjo coeficientes no finitos; no se exporta.")
}

if (!isTRUE(modelo_logistica_final$converged)) {
  stop("La logística no ha convergido; no se exporta.")
}

prob_clas_test <- as.numeric(
  stats::plogis(X_clas_test %*% modelo_logistica_final$coefficients)
)

pred_clas_test <- as.integer(
  prob_clas_test >= UMBRAL_CLASIFICACION
)

tp <- sum(pred_clas_test == 1 & y_clas_test == 1)
tn <- sum(pred_clas_test == 0 & y_clas_test == 0)
fp <- sum(pred_clas_test == 1 & y_clas_test == 0)
fn <- sum(pred_clas_test == 0 & y_clas_test == 1)

precision_test <- if (tp + fp > 0) tp / (tp + fp) else 0
recall_test <- if (tp + fn > 0) tp / (tp + fn) else 0
especificidad_test <- if (tn + fp > 0) tn / (tn + fp) else 0

f1_test <- if (precision_test + recall_test > 0) {
  2 * precision_test * recall_test /
    (precision_test + recall_test)
} else {
  0
}

metricas_clasificacion_test_final <- data.frame(
  modelo = "Regresion logistica",
  filas_test = length(y_clas_test),
  prevalencia_pct = mean(y_clas_test == 1) * 100,
  AUC = calcular_auc_final(y_clas_test, prob_clas_test),
  AP = calcular_ap_final(y_clas_test, prob_clas_test),
  accuracy = mean(pred_clas_test == y_clas_test),
  precision = precision_test,
  recall = recall_test,
  especificidad = especificidad_test,
  F1 = f1_test,
  umbral = UMBRAL_CLASIFICACION,
  TP = tp, TN = tn, FP = fp, FN = fn
)

cat("\n===== MÉTRICAS DE CLASIFICACIÓN EN TEST FINAL =====\n")
print(metricas_clasificacion_test_final, row.names = FALSE, digits = 4)

# ------------------------------------------------------------
# 3. REGRESIÓN: GBM con objetivo log1p
# Solo aprende de vuelos etiquetados con retraso >= 15 min.
# ------------------------------------------------------------

cat("\n===== ENTRENAMIENTO DE REGRESIÓN =====\n")

datos_reg_dev <- datos_desarrollo %>%
  dplyr::filter(retraso_15_min == 1) %>%
  dplyr::select(
    dplyr::all_of(variables_predictoras),
    retraso_minutos
  )

datos_reg_test <- datos_test_final %>%
  dplyr::select(
    dplyr::all_of(variables_predictoras),
    retraso_minutos,
    retraso_15_min
  )

if (nrow(datos_reg_dev) == 0) {
  stop("No hay vuelos retrasados para entrenar la regresión.")
}

prep_reg_final <- preparar_categorias_temporal(
  datos_reg_dev,
  datos_reg_dev,
  datos_reg_test,
  variables_categoricas = variables_categoricas,
  frecuencia_minima = 2
)

x_reg_dev_completo <- crear_matriz_predictores(
  prep_reg_final$train,
  formula_x
)

columnas_x_reg_final <- columnas_variables(x_reg_dev_completo)

if (length(columnas_x_reg_final) == 0) {
  stop("No hay variables válidas para entrenar la regresión.")
}

x_reg_dev <- x_reg_dev_completo[
  , columnas_x_reg_final, drop = FALSE
]

x_reg_test <- crear_matriz_predictores(
  prep_reg_final$test,
  formula_x,
  columnas_x_reg_final
)

y_reg_dev <- as.numeric(prep_reg_final$train$retraso_minutos)

if (any(!is.finite(y_reg_dev)) || any(y_reg_dev < 0)) {
  stop("La variable objetivo de regresión contiene valores no válidos.")
}

datos_gbm_reg_final <- data.frame(
  retraso_log = log1p(y_reg_dev),
  as.data.frame(x_reg_dev),
  check.names = TRUE
)

set.seed(2026)

modelo_gbm_regresion_final <- gbm::gbm(
  retraso_log ~ .,
  data = datos_gbm_reg_final,
  distribution = "gaussian",
  n.trees = N_TREES_REG,
  interaction.depth = 3,
  shrinkage = 0.03,
  n.minobsinnode = 20,
  bag.fraction = 0.5,
  train.fraction = 1,
  verbose = FALSE
)

pred_log_test <- as.numeric(
  predict(
    modelo_gbm_regresion_final,
    newdata = as.data.frame(x_reg_test),
    n.trees = N_TREES_REG,
    type = "response"
  )
)

# Volver a minutos y evitar predicciones por debajo del objetivo mínimo.
pred_minutos_test <- pmax(15, expm1(pred_log_test))

# Métricas de minutos exclusivamente sobre retrasos reales etiquetados
# como >= 15 min en TEST.
idx_retrasados_test <- which(datos_test_final$retraso_15_min == 1)

real_minutos_test <- as.numeric(
  datos_test_final$retraso_minutos[idx_retrasados_test]
)

pred_minutos_retrasados_test <- pred_minutos_test[
  idx_retrasados_test
]

ok_reg <- is.finite(real_minutos_test) &
  is.finite(pred_minutos_retrasados_test)

if (!any(ok_reg)) {
  stop("No hay predicciones válidas para evaluar la regresión en TEST.")
}

metricas_regresion_test_final <- data.frame(
  modelo = "GBM log1p",
  filas_test_retrasadas = sum(ok_reg),
  MAE_minutos = mean(abs(
    real_minutos_test[ok_reg] -
      pred_minutos_retrasados_test[ok_reg]
  )),
  RMSE_minutos = sqrt(mean(
    (real_minutos_test[ok_reg] -
       pred_minutos_retrasados_test[ok_reg])^2
  ))
)

cat("\n===== MÉTRICAS DE REGRESIÓN EN TEST FINAL =====\n")
print(metricas_regresion_test_final, row.names = FALSE, digits = 4)

# ------------------------------------------------------------
# 4. Crear el artefacto final, con reglas y columnas necesarias
# para reconstruir las matrices de predicción desde la API.
# ------------------------------------------------------------

artefacto_aeropredict_final <- list(
  version = "1.0.0",
  fecha_corte_datos = as.character(fecha_corte_datos),
  filas_desarrollo = nrow(datos_desarrollo),
  fecha_inicio_test = as.character(min(datos_test_final$fecha_vuelo)),
  fecha_fin_test = as.character(max(datos_test_final$fecha_vuelo)),
  variables_predictoras = variables_predictoras,
  variables_categoricas = variables_categoricas,
  formula_predictoras = formula_x,
  frecuencia_minima_categoria = 2,
  definicion_objetivo = paste(
    "retraso_15_min = 1 si retraso_minutos >= 15;",
    "retraso_minutos procede de estimated_local - scheduled_local",
    "y puede no representar un retraso real observado."
  ),
  clasificacion = list(
    algoritmo = "Regresion logistica binomial",
    coeficientes = modelo_logistica_final$coefficients,
    columnas_matriz = columnas_x_clas_final,
    columnas_diseno = columnas_glm_final,
    umbral = UMBRAL_CLASIFICACION,
    reglas_categorias = prep_clas_final$reglas
  ),
  regresion = list(
    algoritmo = "GBM gaussiano sobre log1p(retraso_minutos)",
    modelo = modelo_gbm_regresion_final,
    n.trees = N_TREES_REG,
    columna_objetivo_transformada = "retraso_log",
    suelo_prediccion_minutos = 15,
    columnas_matriz = columnas_x_reg_final,
    reglas_categorias = prep_reg_final$reglas
  ),
  metricas_test = list(
    clasificacion = metricas_clasificacion_test_final,
    regresion = metricas_regresion_test_final
  )
)

saveRDS(
  artefacto_aeropredict_final,
  file = "aeropredict_modelos_finales_v1.rds"
)

write.csv(
  metricas_clasificacion_test_final,
  "aeropredict_metricas_clasificacion_final.csv",
  row.names = FALSE,
  fileEncoding = "UTF-8"
)

write.csv(
  metricas_regresion_test_final,
  "aeropredict_metricas_regresion_final.csv",
  row.names = FALSE,
  fileEncoding = "UTF-8"
)

write.csv(
  data.frame(
    scheduled_local = as.character(datos_test_final$scheduled_local),
    flight_number = datos_test_final$flight_number,
    aena_airport_iata = datos_test_final$aena_airport_iata,
    retraso_15_min_real = y_clas_test,
    probabilidad_retraso = prob_clas_test,
    prediccion_retraso_15_min = pred_clas_test
  ),
  "aeropredict_predicciones_clasificacion_final.csv",
  row.names = FALSE,
  fileEncoding = "UTF-8"
)

cat("\n===== ARTEFACTO FINAL GUARDADO =====\n")
print(file.info("aeropredict_modelos_finales_v1.rds")[, c("size", "mtime")])
cat("RDS: aeropredict_modelos_finales_v1.rds\n")
cat("CSV de métricas y predicciones guardados.\n")

# Verificación básica de carga y estructura
artefacto_verificado <- readRDS("aeropredict_modelos_finales_v1.rds")

stopifnot(
  !is.null(artefacto_verificado$clasificacion$coeficientes),
  !is.null(artefacto_verificado$regresion$modelo),
  length(artefacto_verificado$clasificacion$columnas_matriz) > 0,
  length(artefacto_verificado$regresion$columnas_matriz) > 0
)

cat("Verificación de lectura del RDS: correcta.\n")

Filas para entrenamiento final: 58668 
Filas reservadas para TEST: 24739 

===== ENTRENAMIENTO DE CLASIFICACIÓN =====

===== MÉTRICAS DE CLASIFICACIÓN EN TEST FINAL =====
              modelo filas_test prevalencia_pct   AUC     AP accuracy precision
 Regresion logistica      24739           24.19 0.638 0.3676   0.7584    0.5326
   recall especificidad      F1 umbral TP    TN FP   FN
 0.008189        0.9977 0.01613    0.5 49 18712 43 5935

===== ENTRENAMIENTO DE REGRESIÓN =====

===== MÉTRICAS DE REGRESIÓN EN TEST FINAL =====
    modelo filas_test_retrasadas MAE_minutos RMSE_minutos
 GBM log1p                  5984       33.21         68.2

===== ARTEFACTO FINAL GUARDADO =====
                                      size               mtime
aeropredict_modelos_finales_v1.rds 6202036 2026-10-23 11:07:22
RDS: aeropredict_modelos_finales_v1.rds
CSV de métricas y predicciones guardados.
Verificación de lectura del RDS: correcta.


# **2. Ajustar el umbral sin tocar el TEST**

In [ ]:
# ============================================================
# AEROPREDICT — CALIBRACIÓN DEL UMBRAL EN VALIDACIÓN
# No se entrena una nueva familia de modelos.
# No se utiliza TEST para seleccionar el umbral.
# ============================================================

stopifnot(
  exists("datos_train"),
  exists("datos_val"),
  exists("variables_predictoras"),
  exists("variables_categoricas"),
  exists("preparar_categorias_temporal"),
  exists("crear_matriz_predictores"),
  exists("formula_x"),
  exists("prob_clas_test"),
  exists("y_clas_test"),
  exists("artefacto_aeropredict_final")
)

# 1. Preparar TRAIN y VALIDACIÓN
train_umbral <- datos_train %>%
  dplyr::select(
    retraso_15_min,
    dplyr::all_of(variables_predictoras)
  )

val_umbral <- datos_val %>%
  dplyr::select(
    retraso_15_min,
    dplyr::all_of(variables_predictoras)
  )

prep_umbral <- preparar_categorias_temporal(
  train_umbral,
  val_umbral,
  val_umbral,
  variables_categoricas = variables_categoricas,
  frecuencia_minima = 2
)

X_train0 <- crear_matriz_predictores(
  prep_umbral$train, formula_x
)

# Retirar columnas constantes en TRAIN.
columnas_validas <- vapply(
  seq_len(ncol(X_train0)),
  function(j) length(unique(X_train0[, j])) > 1,
  logical(1)
)

columnas_umbral <- colnames(X_train0)[columnas_validas]
X_train <- X_train0[, columnas_umbral, drop = FALSE]

X_val <- crear_matriz_predictores(
  prep_umbral$validacion,
  formula_x,
  columnas_umbral
)

y_train_umbral <- as.integer(
  as.character(prep_umbral$train$retraso_15_min)
)

y_val_umbral <- as.integer(
  as.character(prep_umbral$validacion$retraso_15_min)
)

# Matrices con intercepto y columnas independientes.
X_train_full <- cbind(Intercept = 1, as.matrix(X_train))
X_val_full <- cbind(Intercept = 1, as.matrix(X_val))

qr_umbral <- qr(X_train_full)
idx_umbral <- qr_umbral$pivot[seq_len(qr_umbral$rank)]
cols_glm <- colnames(X_train_full)[idx_umbral]

X_train_glm <- X_train_full[, cols_glm, drop = FALSE]
X_val_glm <- X_val_full[, cols_glm, drop = FALSE]

modelo_umbral <- suppressWarnings(
  stats::glm.fit(
    x = X_train_glm,
    y = y_train_umbral,
    family = stats::binomial(),
    control = stats::glm.control(maxit = 100)
  )
)

if (!isTRUE(modelo_umbral$converged) ||
    any(!is.finite(modelo_umbral$coefficients))) {
  stop("La logística de VALIDACIÓN no convergió correctamente.")
}

score_val_umbral <- as.numeric(
  plogis(X_val_glm %*% modelo_umbral$coefficients)
)

if (any(!is.finite(score_val_umbral))) {
  stop("Hay probabilidades no válidas en VALIDACIÓN.")
}

# 2. Evaluar umbrales solo en VALIDACIÓN
calcular_operacion <- function(real, score, umbral) {
  pred <- as.integer(score >= umbral)

  tp <- sum(pred == 1 & real == 1)
  tn <- sum(pred == 0 & real == 0)
  fp <- sum(pred == 1 & real == 0)
  fn <- sum(pred == 0 & real == 1)

  precision <- if (tp + fp > 0) tp / (tp + fp) else 0
  recall <- if (tp + fn > 0) tp / (tp + fn) else 0
  especificidad <- if (tn + fp > 0) tn / (tn + fp) else 0
  f1 <- if (precision + recall > 0) {
    2 * precision * recall / (precision + recall)
  } else {
    0
  }

  data.frame(
    umbral = umbral,
    precision = precision,
    recall = recall,
    especificidad = especificidad,
    F1 = f1,
    porcentaje_alertas = mean(pred == 1) * 100
  )
}

umbrales <- seq(0.005, 0.50, by = 0.005)

tabla_umbrales <- do.call(
  rbind,
  lapply(umbrales, function(u) {
    calcular_operacion(y_val_umbral, score_val_umbral, u)
  })
)

# Maximizar F1; desempatar por precisión y luego por recall.
tabla_ordenada <- tabla_umbrales[
  order(
    -tabla_umbrales$F1,
    -tabla_umbrales$precision,
    -tabla_umbrales$recall
  ),
]

umbral_elegido <- tabla_ordenada$umbral[1]

cat("\n===== VALIDACIÓN: UMBRAL 0,5 =====\n")
print(
  calcular_operacion(y_val_umbral, score_val_umbral, 0.5),
  row.names = FALSE,
  digits = 4
)

cat("\n===== VALIDACIÓN: UMBRAL ELEGIDO POR F1 =====\n")
print(
  calcular_operacion(y_val_umbral, score_val_umbral, umbral_elegido),
  row.names = FALSE,
  digits = 4
)

# 3. Aplicar el umbral elegido a las probabilidades TEST ya calculadas.
# No se busca ni se selecciona el umbral usando las etiquetas TEST.
operacion_test <- calcular_operacion(
  y_clas_test,
  prob_clas_test,
  umbral_elegido
)

pred_test_umbral <- as.integer(prob_clas_test >= umbral_elegido)

metricas_clasificacion_test_final$accuracy <- mean(
  pred_test_umbral == y_clas_test
)
metricas_clasificacion_test_final$precision <- operacion_test$precision
metricas_clasificacion_test_final$recall <- operacion_test$recall
metricas_clasificacion_test_final$especificidad <- operacion_test$especificidad
metricas_clasificacion_test_final$F1 <- operacion_test$F1
metricas_clasificacion_test_final$umbral <- umbral_elegido
metricas_clasificacion_test_final$TP <- sum(
  pred_test_umbral == 1 & y_clas_test == 1
)
metricas_clasificacion_test_final$TN <- sum(
  pred_test_umbral == 0 & y_clas_test == 0
)
metricas_clasificacion_test_final$FP <- sum(
  pred_test_umbral == 1 & y_clas_test == 0
)
metricas_clasificacion_test_final$FN <- sum(
  pred_test_umbral == 0 & y_clas_test == 1
)

# Actualizar umbral y métricas dentro del artefacto ya guardado.
artefacto_aeropredict_final$clasificacion$umbral <- umbral_elegido
artefacto_aeropredict_final$metricas_test$clasificacion <-
  metricas_clasificacion_test_final

saveRDS(
  artefacto_aeropredict_final,
  "aeropredict_modelos_finales_v1.rds"
)

write.csv(
  metricas_clasificacion_test_final,
  "aeropredict_metricas_clasificacion_final.csv",
  row.names = FALSE,
  fileEncoding = "UTF-8"
)

cat("\n===== MÉTRICAS TEST CON UMBRAL ELEGIDO EN VALIDACIÓN =====\n")
print(metricas_clasificacion_test_final, row.names = FALSE, digits = 4)

cat("\nUmbral guardado en RDS:", umbral_elegido, "\n")
cat("Artefacto actualizado correctamente.\n")


===== VALIDACIÓN: UMBRAL 0,5 =====
 umbral precision  recall especificidad      F1 porcentaje_alertas
    0.5   0.07336 0.09428        0.9201 0.08251              8.079

===== VALIDACIÓN: UMBRAL ELEGIDO POR F1 =====
 umbral precision recall especificidad     F1 porcentaje_alertas
    0.1   0.09392 0.8721        0.4356 0.1696              58.37

===== MÉTRICAS TEST CON UMBRAL ELEGIDO EN VALIDACIÓN =====
              modelo filas_test prevalencia_pct   AUC     AP accuracy precision
 Regresion logistica      24739           24.19 0.638 0.3676   0.6035    0.3266
 recall especificidad     F1 umbral   TP    TN   FP   FN
 0.6019        0.6039 0.4234    0.1 3602 11327 7428 2382

Umbral guardado en RDS: 0.1 
Artefacto actualizado correctamente.


In [ ]:
# ============================================================
# AEROPREDICT — PESOS E IMPORTANCIA DE LOS MODELOS FINALES
# No entrena modelos nuevos.
# ============================================================

artefacto <- readRDS("aeropredict_modelos_finales_v1.rds")

# 1. Coeficientes y odds ratios de la logística final.
coef_final <- artefacto$clasificacion$coeficientes

grupo_variable <- function(v) {
  if (v %in% c("Intercept", "(Intercept)")) return("Intercepto")
  if (v == "temperatura_metar") return("temperatura_metar")
  if (v == "punto_rocio_metar") return("punto_rocio_metar")
  if (v == "relh") return("relh")
  if (startsWith(v, "hora_vuelo")) return("hora_vuelo")
  if (startsWith(v, "dia_semana")) return("dia_semana")
  if (startsWith(v, "airline_iata")) return("airline_iata")
  if (startsWith(v, "other_airport_iata")) return("other_airport_iata")
  "otra"
}

tabla_coeficientes <- data.frame(
  termino = names(coef_final),
  variable = vapply(names(coef_final), grupo_variable, character(1)),
  coeficiente_logit = as.numeric(coef_final),
  odds_ratio = exp(as.numeric(coef_final))
)

cat("\n===== COEFICIENTES DE LA LOGÍSTICA FINAL =====\n")
print(tabla_coeficientes, row.names = FALSE, digits = 4)

# 2. Importancia por permutación en VALIDACIÓN.
# Se utiliza el modelo entrenado en TRAIN para la calibración
# del umbral; no se emplea el TEST para calcular esta importancia.

if (exists("modelo_umbral") &&
    exists("X_val_glm") &&
    exists("y_val_umbral")) {

  X <- as.matrix(X_val_glm)
  beta <- as.numeric(modelo_umbral$coefficients)
  y <- as.integer(y_val_umbral)

  stopifnot(ncol(X) == length(beta), nrow(X) == length(y))

  auc_local <- function(y, p) {
    n1 <- sum(y == 1)
    n0 <- sum(y == 0)
    if (n1 == 0 || n0 == 0) return(NA_real_)
    r <- rank(p, ties.method = "average")
    (sum(r[y == 1]) - n1 * (n1 + 1) / 2) / (n1 * n0)
  }

  ap_local <- function(y, p) {
    if (sum(y == 1) == 0) return(NA_real_)
    o <- order(p, decreasing = TRUE)
    yy <- y[o]
    mean((cumsum(yy) / seq_along(yy))[yy == 1])
  }

  grupos_columnas <- vapply(
    colnames(X), grupo_variable, character(1)
  )

  score_base <- plogis(drop(X %*% beta))
  auc_base <- auc_local(y, score_base)
  ap_base <- ap_local(y, score_base)

  set.seed(2026)
  B <- 30
  variables <- setdiff(unique(grupos_columnas),
                       c("Intercepto", "otra"))

  importancia <- lapply(variables, function(v) {
    cols <- which(grupos_columnas == v)
    caida_auc <- numeric(B)
    caida_ap <- numeric(B)

    for (b in seq_len(B)) {
      Xp <- X
      perm <- sample.int(nrow(Xp))
      Xp[, cols] <- X[perm, cols, drop = FALSE]

      p_perm <- plogis(drop(Xp %*% beta))
      caida_auc[b] <- auc_base - auc_local(y, p_perm)
      caida_ap[b] <- ap_base - ap_local(y, p_perm)
    }

    data.frame(
      variable = v,
      caida_AUC_media = mean(caida_auc, na.rm = TRUE),
      caida_AP_media = mean(caida_ap, na.rm = TRUE)
    )
  })

  tabla_importancia_logistica <- do.call(rbind, importancia)
  tabla_importancia_logistica <- tabla_importancia_logistica[
    order(-tabla_importancia_logistica$caida_AP_media), ]

  cat("\n===== IMPORTANCIA POR PERMUTACIÓN: LOGÍSTICA =====\n")
  print(tabla_importancia_logistica, row.names = FALSE, digits = 4)

} else {
  cat(
    "\nNo se encontraron los objetos de validación de la logística.",
    "No se ha calculado importancia por permutación.\n"
  )
}

# 3. Importancia relativa del GBM logarítmico final.
imp_reg <- summary(
  artefacto$regresion$modelo,
  n.trees = artefacto$regresion$n.trees,
  plotit = FALSE
)

imp_reg$variable <- vapply(
  as.character(imp_reg$var),
  grupo_variable,
  character(1)
)

imp_reg_agrupada <- aggregate(
  rel.inf ~ variable,
  data = imp_reg,
  FUN = sum
)

imp_reg_agrupada <- imp_reg_agrupada[
  order(-imp_reg_agrupada$rel.inf), ]

cat("\n===== IMPORTANCIA DEL GBM DE REGRESIÓN FINAL =====\n")
print(imp_reg_agrupada, row.names = FALSE, digits = 4)


===== COEFICIENTES DE LA LOGÍSTICA FINAL =====
                termino           variable coeficiente_logit odds_ratio
              Intercept         Intercepto          -3.64745  2.606e-02
            hora_vuelo1         hora_vuelo          -2.29398  1.009e-01
            hora_vuelo2         hora_vuelo          -1.94461  1.430e-01
            hora_vuelo3         hora_vuelo           0.33846  1.403e+00
            hora_vuelo4         hora_vuelo           2.80050  1.645e+01
            hora_vuelo5         hora_vuelo          -2.11338  1.208e-01
            hora_vuelo6         hora_vuelo          -3.16524  4.220e-02
            hora_vuelo7         hora_vuelo          -2.85751  5.741e-02
            hora_vuelo8         hora_vuelo          -2.26584  1.037e-01
            hora_vuelo9         hora_vuelo          -1.81312  1.631e-01
           hora_vuelo10         hora_vuelo          -1.44658  2.354e-01
           hora_vuelo11         hora_vuelo          -1.03567  3.550e-01
           hora_